In [1]:
# --- Master switches (pre-run, resilient) ---
# Turn everything ON early, even before CFG exists, for modules that read CONFIG.
CONFIG = globals().get("CONFIG", {})
CONFIG.update({
    "RUN_UI": True,
    "RUN_SPC": True,
    "RUN_GATE": True,
    "RUN_MEDS": True,
    "PHASE2_BUNDLE": True,
})
globals()["CONFIG"] = CONFIG
print("CONFIG master switches set →", {k: CONFIG[k] for k in ["RUN_UI","RUN_SPC","RUN_GATE","RUN_MEDS","PHASE2_BUNDLE"]})


CONFIG master switches set → {'RUN_UI': True, 'RUN_SPC': True, 'RUN_GATE': True, 'RUN_MEDS': True, 'PHASE2_BUNDLE': True}


In [2]:
# --- Early utilities: robust event logging ---
from pathlib import Path as _Path
from datetime import datetime as _dt, timezone as _tz
import json as _json

def _append_event(ev: dict):
    """
    Append an event to the event log.
    - If CFG is available and has event_log_path, use it.
    - Otherwise, fall back to /kaggle/working/event_log.jsonl (or ./event_log.jsonl).
    """
    # Determine target path
    try:
        p = _Path(str(CFG.event_log_path))  # type: ignore[name-defined]
    except Exception:
        # Fallbacks if CFG not defined yet
        if _Path("/kaggle/working").exists():
            p = _Path("/kaggle/working/event_log.jsonl")
        else:
            p = _Path("event_log.jsonl")
    # Ensure file/dir
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch(exist_ok=True)
    # Compose event
    ev = {"ts": _dt.now(_tz.utc).isoformat(), **(ev or {})}
    with p.open("a", encoding="utf-8") as _fp:
        _fp.write(_json.dumps(ev, ensure_ascii=False) + "\n")


In [3]:
# --- UI argument shims: tracker, get_state, get_actions, critic ---
from types import SimpleNamespace
import pandas as _pd

# Try to detect a tracker-like object already in scope
_tracker = None
for name in ["tracker_core", "tracker", "core_tracker"]:
    if name in globals():
        _tracker = globals()[name]
        break

class _State:
    def __init__(self, feats=None):
        self._feats = dict(feats or {})
    def feature_dict(self):
        return dict(self._feats)
    def touch_now(self, now):
        # Reset "since_vitals_min" if present; else set it to 0
        self._feats["since_vitals_min"] = 0

def _derive_state_from_data():
    # Best-effort: if a feature builder exists, use it; otherwise synthesize minimal features
    feats = {}
    # If your pipeline exposes a feature extractor, prefer it here
    if "predict_one" in globals():
        try:
            # Unpack a minimal vitals context if available
            feats["since_vitals_min"] = 30  # neutral default if unknown
        except Exception:
            feats["since_vitals_min"] = 30
    else:
        feats["since_vitals_min"] = 30
    return _State(feats)

def get_state():
    return _derive_state_from_data()

def get_actions(state):
    actions = []
    # If a gate_engine is available, attempt to fetch actions from common naming
    ge = globals().get("gate_engine", None)
    if ge is not None:
        for cand in ["propose_actions", "propose", "actions"]:
            fn = getattr(ge, cand, None)
            if callable(fn):
                try:
                    out = fn(state) if fn.__code__.co_argcount >= 1 else fn()
                    # Normalize to a list of dicts with id/label
                    if isinstance(out, dict):
                        out = [out]
                    for i, a in enumerate(out or []):
                        if isinstance(a, dict):
                            actions.append({"id": a.get("id", f"a{i}"), "label": a.get("label", a.get("name", f"Action {i}"))})
                        else:
                            actions.append({"id": f"a{i}", "label": str(a)})
                    break
                except Exception:
                    pass
    # Fallback: provide a harmless no-op action so UI renders
    if not actions:
        actions = [{"id":"noop","label":"Reassess in 30 min"}]
    return actions

class _Critic:
    def score(self, state, actions):
        import numpy as np
        n = len(actions)
        # Neutral, sorted-friendly scores
        p = np.full(n, 0.5)
        benefit = np.linspace(0.2, 0.8, n) if n else np.array([])
        burden = np.linspace(0.8, 0.2, n) if n else np.array([])
        return p, benefit, burden

critic = globals().get("critic", None) or _Critic()

# Provide a simple tracker wrapper if none present, exposing movement_stats() used by UI
if _tracker is None:
    class _SimpleTracker:
        def movement_stats(self):
            # Minimal dataframe-based stats if vitals/ed tables exist
            ed = globals().get("ed", None)
            if isinstance(ed, _pd.DataFrame) and not ed.empty:
                per_eq = _pd.DataFrame({"equip_id":[f"E{i}" for i in range(5)], "moves":[5,3,7,2,4]})
                routes = _pd.DataFrame({"route":[("A","B"),("B","C")],"count":[10,6]})
            else:
                per_eq = _pd.DataFrame(columns=["equip_id","moves"])
                routes = _pd.DataFrame(columns=["route","count"])
            return {"moves_per_equipment": per_eq, "routes": routes}
    _tracker = _SimpleTracker()

tracker = _tracker


In [4]:
# --- SPC entrypoint (robust & tidy) ---
def _fallback_run_spc() -> None:
    """Minimal SPC view used only if no real SPC entrypoint exists."""
    import streamlit as st
    import pandas as pd

    st.header("SPC — Quick View")
    for name in ("ed", "dx", "meds"):
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            st.subheader(f"{name.upper()} sample")
            st.dataframe(obj.head(50), use_container_width=True, height=240)
    st.caption("SPC fallback: replace with your domain SPC charts.")

# Prefer existing entrypoints; otherwise provide an alias or fallback.
if "run_spc" not in globals():
    if "spc_dashboard" in globals():
        run_spc = spc_dashboard          # alias real dashboard if present
    else:
        run_spc = _fallback_run_spc      # safe fallback


In [5]:
# --- Settings (inline, no external file) ---
from __future__ import annotations
import os
from dataclasses import dataclass, field, replace
from pathlib import Path

def detect_base() -> Path:
    if (env := os.getenv("DATA_ROOT")):
        p = Path(env); p.mkdir(parents=True, exist_ok=True); return p
    if os.path.exists("/kaggle/working"): return Path("/kaggle/working")
    if os.path.exists("/content"):         return Path("/content")
    return Path("/mnt/data")

@dataclass(frozen=True)
class Settings:
    # your fields + defaults here
    data_root: Path = field(default_factory=detect_base)
    model_bundle_path: Path | None = None
    skip_model_discovery: bool = False
    # ... keep the rest of your attributes/properties exactly as before ...

    # --- derived paths ---
    @property
    def equipment_status_path(self) -> Path:     return self.data_root / "equipment_status.csv"
    @property
    def equipment_moves_log_path(self) -> Path:  return self.data_root / "moves_log.csv"
    @property
    def sop_registry_path(self) -> Path:         return self.data_root / "sop_registry.csv"
    @property
    def qr_output_dir(self) -> Path:             return self.data_root / "qrs"
    @property
    def event_log_path(self) -> Path:            return self.data_root / "event_log.jsonl"

    def ensure_fs(self) -> None:
        self.qr_output_dir.mkdir(parents=True, exist_ok=True)
        self.event_log_path.parent.mkdir(parents=True, exist_ok=True)

# --- Runtime config + bundle attach (CFG is single source of truth) ---
CFG = Settings()
CFG.ensure_fs()

DATASET = "ed-pipeline-bundle-ui"
SRC = f"/kaggle/input/{DATASET}/ed_phase2_model_thr_patched(1).joblib"
DST = "/kaggle/working/ed_phase2_model_thr_patched(1).joblib"

import os, shutil
from pathlib import Path

assert os.path.exists(SRC), f"Not found: {SRC}"
Path(os.path.dirname(DST)).mkdir(parents=True, exist_ok=True)
if not os.path.exists(DST):
    shutil.copy2(SRC, DST)

CFG = replace(CFG, model_bundle_path=Path(DST), skip_model_discovery=True)

print("BASE =", CFG.data_root)
print("Event log path =", CFG.event_log_path)
print("Bundle ready →", CFG.model_bundle_path)


BASE = /kaggle/working
Event log path = /kaggle/working/event_log.jsonl
Bundle ready → /kaggle/working/ed_phase2_model_thr_patched(1).joblib


In [6]:
# --- Bundle loader (CFG-first, single source of truth) ---
from joblib import load
import os

candidates = [
    getattr(CFG, "model_bundle_path", None),
    "/kaggle/working/ed_phase2_model_thr_patched(1).joblib",
    "/kaggle/input/ed-pipeline-bundle-ui/ed_phase2_model_thr_patched(1).joblib",
    os.getenv("MODEL_BUNDLE_PATH"),
]
bundle_path = next((p for p in candidates if p and os.path.exists(p)), None)
if not bundle_path:
    tried = [p for p in candidates if p]
    raise FileNotFoundError("Model bundle not found. Tried:\n" + "\n".join(f" - {p}" for p in tried))

print("Using model bundle →", bundle_path)
bundle = load(bundle_path)

pipe = bundle.get("pipeline") or bundle.get("model")
cal  = bundle.get("calibrator")
THR  = float(bundle.get("threshold", 0.5))
FEAT = list(bundle.get("features") or [])

print("Loaded keys:", list(bundle.keys()))
print("Pipeline:", type(pipe).__name__ if pipe is not None else None)
print("Calibrator:", type(cal).__name__ if cal is not None else None)

Using model bundle → /kaggle/working/ed_phase2_model_thr_patched(1).joblib
Loaded keys: ['pipeline', 'calibrator', 'threshold', 'features', 'label', 'model_kind', 'created_utc']
Pipeline: Pipeline
Calibrator: IsotonicRegression


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator SimpleImputer from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator StandardScaler from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelBinarizer from version 1.1.3 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For mor

In [7]:
# === ML Wrapper (self-initializing; CFG-first) ===
import os
import numpy as np
import pandas as pd
from joblib import load

def _ensure_model_loaded():
    """Ensure globals: pipe, cal, THR, FEAT."""
    g = globals()
    needed = all(k in g for k in ("pipe", "cal", "THR", "FEAT"))
    if needed:
        return
    # Resolve bundle path (prefer CFG)
    bundle_path = getattr(g.get("CFG", None), "model_bundle_path", None)
    if not (bundle_path and os.path.exists(bundle_path)):
        # fallback to known locations / env
        candidates = [
            bundle_path,
            "/kaggle/working/ed_phase2_model_thr_patched(1).joblib",
            "/kaggle/input/ed-pipeline-bundle-ui/ed_phase2_model_thr_patched(1).joblib",
            os.getenv("MODEL_BUNDLE_PATH"),
        ]
        bundle_path = next((p for p in candidates if p and os.path.exists(p)), None)
    assert bundle_path and os.path.exists(bundle_path), f"Bundle missing at {bundle_path}"
    bundle = load(bundle_path)
    g["pipe"] = bundle.get("pipeline") or bundle.get("model")
    g["cal"]  = bundle.get("calibrator")
    g["THR"]  = float(bundle.get("threshold", 0.5))
    g["FEAT"] = list(bundle.get("features") or [])

_ensure_model_loaded()  # make sure FEAT/THR exist before defining functions

def _predict_raw(X: pd.DataFrame) -> np.ndarray:
    if hasattr(pipe, "predict_proba"):
        return np.asarray(pipe.predict_proba(X)[:, 1]).ravel()
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X)).ravel()
        return 1 / (1 + np.exp(-z))
    y = np.asarray(pipe.predict(X)).ravel()
    return np.clip(y.astype(float), 0.0, 1.0)

def score_proba(df: pd.DataFrame) -> np.ndarray:
    X = df.reindex(columns=FEAT, fill_value=0) if FEAT else df
    p = _predict_raw(X)
    if cal is not None:
        try:
            p = np.asarray(cal.transform(p)).ravel()  # isotonic expects 1-D
        except Exception:
            pass
    return np.clip(p, 0.0, 1.0)

def predict_one(row: dict) -> dict:
    p = float(score_proba(pd.DataFrame([row]))[0])
    return {"p": p, "y": int(p >= THR), "thr": THR}

print(f"ML wrapper ready | features={len(FEAT)} | threshold={THR}")


ML wrapper ready | features=16 | threshold=0.9988444286248084


In [8]:
# --- minimal scorer + smoke test ---
import numpy as np
import pandas as pd

FEATURES = bundle.get("features") or []
THRESH   = float(bundle.get("threshold", 0.5))
pipe     = bundle["pipeline"]
cal      = bundle.get("calibrator")  # may be None

def predict_proba_df(df: pd.DataFrame) -> np.ndarray:
    X = df.reindex(columns=FEATURES, fill_value=0) if FEATURES else df
    raw = pipe.predict_proba(X)[:, 1] if hasattr(pipe, "predict_proba") else pipe.predict(X)
    if cal is not None:
        # Ensure 2D for calibrator (Isotonic expects (n_samples,))
        raw = np.asarray(raw).ravel()
        raw = cal.transform(raw)
    return np.clip(raw, 0, 1)

def predict_label_df(df: pd.DataFrame, thr: float = THRESH) -> np.ndarray:
    return (predict_proba_df(df) >= thr).astype(int)

# smoke: one all-zero row with correct columns
smoke_df = pd.DataFrame([[0]*len(FEATURES)], columns=FEATURES)
proba = predict_proba_df(smoke_df)
label = predict_label_df(smoke_df)

print(f"features: {len(FEATURES)} | threshold: {THRESH}")
print("proba[0]:", float(proba[0]))
print("label[0]:", int(label[0]))


features: 16 | threshold: 0.9988444286248084
proba[0]: 0.0
label[0]: 0


In [9]:
# Simple import handler (deterministic; known paths)
import sys, os
from importlib import import_module

# Known locations
CORE_DIR = "/kaggle/input/tracker-core"
OPS_PKG  = "/kaggle/input/edtracker-ops-pg"    # edtracker_ops_pkg/
UI_PKG   = "/kaggle/input/edtracker-ui-pkg"    # edtracker_ui_pkg/
OPS_INTEL_DIR = "/kaggle/input/ops-intel"      # ops_intel.py

for p in [OPS_PKG, UI_PKG, CORE_DIR, OPS_INTEL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Imports
tracker_core = import_module("tracker_core")
try:    edtracker_ops_pkg = import_module("edtracker_ops_pkg")
except: edtracker_ops_pkg = None
try:    edtracker_ui_pkg  = import_module("edtracker_ui_pkg")
except: edtracker_ui_pkg  = None
try:    ops_intel         = import_module("ops_intel")
except: ops_intel         = None

# Canonical aliases (optional)
sys.modules["tracker_core"] = tracker_core
sys.modules.setdefault("edtracker.core.tracker_core", tracker_core)

print("tracker_core ready:", hasattr(tracker_core, "TrackerService"))


tracker_core ready: True


In [10]:

# WorkflowState invariant: defined before use; exposes required methods; timers/backlogs/alerts intact
from dataclasses import dataclass, field
from typing import Any, Dict, List
import pandas as pd

@dataclass
class WorkflowState:
    role: str
    state: Dict[str, Any] = field(default_factory=dict)
    timers: Dict[str, float] = field(default_factory=dict)
    backlogs: Dict[str, List[Any]] = field(default_factory=dict)
    alerts: List[str] = field(default_factory=list)

    def touch_now(self, ts: pd.Timestamp):
        # update an example timer
        self.timers["last_touch_epoch"] = float(ts.value) / 1e9

    def feature_dict(self) -> Dict[str, Any]:
        # safe, extendable
        fd = {
            "role": self.role,
            "since_vitals_min": self.state.get("since_vitals_min", 0.0),
            "alerts_count": len(self.alerts),
        }
        # pass through any extra scalar features
        for k,v in self.state.items():
            if isinstance(v,(int,float,str)) and k not in fd:
                fd[k]=v
        return fd

    def update_state_from_event(self, event: Dict[str, Any]):
        # naive: merge event into state; track backlog
        self.state.update(event)
        self.backlogs.setdefault("events", []).append(event)

    def apply_event_log(self, events: List[Dict[str, Any]]):
        for ev in events:
            self.update_state_from_event(ev)


In [11]:

# TinyCritics uses WorkflowState.feature_dict(); cold-start safe (no transform before fit)
import numpy as np

class TinyCritics:
    def __init__(self):
        self._fitted = False

    def fit(self, states, actions, rewards):
        # No-op fit to keep cold-start safe
        self._fitted = True
        return self

    def score(self, state: WorkflowState, actions: List[Dict[str,str]]):
        fd = state.feature_dict()
        n = len(actions)
        # Deterministic, bounded scores in [0,1]
        base = 0.5
        p = np.full(n, base, dtype=float)
        bonuses = np.zeros(n, dtype=float)
        uncertainty = np.full(n, 0.1, dtype=float)
        return p, bonuses, uncertainty


In [12]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd, numpy as np
import hashlib

def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default

def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)

@dataclass
class EquipmentRecord:
    equip_id: str; name: str=""; location: str=""; status: str=""; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None
    def to_row(self)->Dict[str,Any]:
        return {"equip_id":self.equip_id,"name":self.name,"location":self.location,"status":self.status,"last_seen":self.last_seen,"battery":self.battery,"confidence":self.confidence}

class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists():
            pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"]).to_csv(self.status_csv, index=False)
    def read(self)->pd.DataFrame:
        try: df=pd.read_csv(self.status_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"])
        if "equip_id" in df.columns: df["equip_id"]=df["equip_id"].astype(str)
        return df
    def upsert(self, rec: EquipmentRecord)->None:
        df=self.read(); row=pd.DataFrame([rec.to_row()])
        if df.empty: df=row
        else:
            mask=(df["equip_id"].astype(str)==str(rec.equip_id))
            if mask.any():
                for col in row.columns: df.loc[mask,col]=row[col].values[0]
            else: df=pd.concat([df,row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)

class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists():
            pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(self.moves_csv, index=False)
    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:
        row=pd.DataFrame([{"equip_id":equip_id,"from":loc_from,"to":loc_to,"ts":ts_iso}])
        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row
        except Exception: df=row
        df.to_csv(self.moves_csv, index=False)
    def read(self)->pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","from","to","ts"])

class SOPRegistry:
    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)
    def read(self)->pd.DataFrame:
        if self.sop_csv.exists():
            try:
                df=pd.read_csv(self.sop_csv)
                for col in ["sop_id","title","pdf_path"]:
                    if col not in df.columns: df[col]=""
                return df
            except Exception: pass
        return pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])

class QRService:
    def __init__(self,out_dir:Path): self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self,payload:str)->str:
        try:
            import qrcode
            h=hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]
            fp=self.out_dir/f"qr_{h}.png"; qrcode.make(payload).save(fp); return str(fp)
        except Exception: return f"[QR fallback] {payload}"
    def decode_file(self,image_bytes:bytes):
        try:
            from PIL import Image; import io
            img=Image.open(io.BytesIO(image_bytes))
            try:
                from pyzbar.pyzbar import decode as zbar_decode
                res=zbar_decode(img); 
                if res: return res[0].data.decode("utf-8","ignore")
            except Exception: pass
        except Exception: pass
        return None

class TrackerService:
    def __init__(self,equipment_repo:EquipmentRepository,moves_repo:MovesLogRepository,sop_registry:SOPRegistry,qr:QRService,config:Any):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG:Any)->"TrackerService":
        return cls(EquipmentRepository(Path(_cfg(CONFIG,"equipment_status_path"))),
                   MovesLogRepository(Path(_cfg(CONFIG,"equipment_moves_log_path"))),
                   SOPRegistry(Path(_cfg(CONFIG,"sop_registry_path"))),
                   QRService(Path(_cfg(CONFIG,"qr_output_dir"))), CONFIG)
    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()
    def log_move(self,equip_id:str,loc_from:str,loc_to:str)->None:
        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()
        name=df[df["equip_id"].astype(str)==str(equip_id)]["name"].iloc[0] if not df.empty and "name" in df.columns else ""
        self.equipment_repo.upsert(EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status="moved",last_seen=ts_iso))
        self.moves_repo.append(equip_id,loc_from or "",loc_to,ts_iso)
    def find_equipment(self,query:str)->pd.DataFrame:
        q=(query or "").strip().lower(); df=self.equipment_repo.read()
        if not q: return df
        return df[df.apply(lambda r:any(q in str(r.get(k,"")).lower() for k in ["equip_id","name","location","status"]),axis=1)]
    def overdue_equipment(self,threshold_minutes:int=120)->pd.DataFrame:
        df=self.equipment_repo.read().copy()
        if df.empty or "last_seen" not in df.columns: return df.iloc[0:0]
        ts=pd.to_datetime(df["last_seen"],errors="coerce",utc=True)
        df["age_min"]=(pd.Timestamp.utcnow().tz_localize("UTC")-ts).dt.total_seconds()/60.0
        return df[df["age_min"]>float(threshold_minutes)].sort_values("age_min",ascending=False)
    def movement_stats(self)->Dict[str,pd.DataFrame]:
        log=self.moves_repo.read()
        if log.empty: return {"moves_per_equipment":log,"routes":log}
        return {"moves_per_equipment":log.groupby("equip_id").size().reset_index(name="moves").sort_values("moves",ascending=False),
                "routes":log.groupby(["from","to"]).size().reset_index(name="count").sort_values("count",ascending=False)}
    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()
    def search_sop(self,query:str)->pd.DataFrame:
        df=self.sop_registry.read().copy(); q=(query or "").strip().lower()
        if df.empty or not q: return df
        cols=[c for c in ["sop_id","title","keywords","version","status"] if c in df.columns]
        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q,na=False)).any(axis=1)
        return df[mask]
    def make_qr(self,payload:str)->str: return self.qr.make(payload)
    def decode_qr_bytes(self,image_bytes:bytes): return self.qr.decode_file(image_bytes)

print("tracker core ready")


tracker core ready


In [13]:
# --- tracker_core alias shim (append-only, no renames) ---
import sys, types
_names = ["TrackerService","QRService","EquipmentRepository","MovesLogRepository","SOPRegistry"]
if "tracker_core" not in sys.modules and all(n in globals() for n in _names):
    _m = types.ModuleType("tracker_core")
    for n in _names:
        setattr(_m, n, globals()[n])
    sys.modules["tracker_core"] = _m
    print("tracker_core alias ready")
else:
    print("tracker_core alias already present or classes missing")


tracker_core alias already present or classes missing


In [14]:
# Merged EquipmentRepository with robust read() (keeps same API/behavior)
from pathlib import Path
import pandas as pd, numpy as np

_EQUIP_COLS = ["equip_id","name","location","status","last_seen","battery","confidence"]

class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv = Path(status_csv)
        self.status_csv.parent.mkdir(parents=True, exist_ok=True)
        if not self.status_csv.exists():
            pd.DataFrame(columns=_EQUIP_COLS).to_csv(self.status_csv, index=False)

    def read(self) -> pd.DataFrame:
        try:
            df = pd.read_csv(self.status_csv)
        except Exception:
            return pd.DataFrame(columns=_EQUIP_COLS)
        # ensure required columns exist (fill missing), normalize types/order
        for col in _EQUIP_COLS:
            if col not in df.columns:
                df[col] = np.nan
        df["equip_id"] = df["equip_id"].astype(str)
        return df[_EQUIP_COLS]

    def upsert(self, rec) -> None:
        df = self.read()
        row = pd.DataFrame([{
            "equip_id": rec.equip_id, "name": rec.name, "location": rec.location,
            "status": rec.status, "last_seen": rec.last_seen,
            "battery": rec.battery, "confidence": rec.confidence
        }])
        if df.empty:
            df = row
        else:
            mask = (df["equip_id"].astype(str) == str(rec.equip_id))
            if mask.any():
                for col in row.columns:
                    df.loc[mask, col] = row[col].values[0]
            else:
                df = pd.concat([df, row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)



In [15]:
import inspect
print("EquipmentRepository.read defined at:", inspect.getsourcefile(EquipmentRepository.read))
print("Is monkey-patched?", getattr(EquipmentRepository.read, "__name__", "") != "read")


EquipmentRepository.read defined at: /tmp/ipykernel_13/3517869032.py
Is monkey-patched? False


In [16]:
from pathlib import Path
from typing import Any, Dict, List
def _slugify(text:str)->str:
    import re; s=re.sub(r"[^a-zA-Z0-9]+","-",text.strip().lower()).strip("-"); return s or "sop"
def load_priority_flows(json_path:str)->Dict[str,Any]:
    import json
    try:
        with open(json_path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception: return {}
print("sop auto ready")


sop auto ready


In [17]:
# Inline SOP surface and optional UI (CFG-first; no external pkgs required)
from typing import Optional, Dict, Any, List
from pathlib import Path
import csv, os

# --- helpers ---------------------------------------------------------------
def ensure_csv(path: Path, headers: List[str]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="") as f:
            csv.DictWriter(f, fieldnames=headers).writeheader()

def load_csv_df(path: Path):
    """Return a pandas DataFrame if pandas is available; else a list[dict]."""
    try:
        import pandas as pd
        return pd.read_csv(path)
    except Exception:
        with Path(path).open(newline="") as f:
            return list(csv.DictReader(f))

def save_csv_row(path: Path, row: Dict[str, Any]) -> None:
    path = Path(path)
    new_file = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if new_file:
            w.writeheader()
        w.writerow(row)

# --- inline SOP surface ----------------------------------------------------
def read_sop_table(path: Path):
    tbl = load_csv_df(path)
    return tbl

def search_sop_table(path: Path, query: str):
    q = (query or "").strip().lower()
    if not q:
        return read_sop_table(path)
    tbl = load_csv_df(path)
    try:
        # pandas path
        import pandas as pd  # noqa: F401
        cols = [c for c in ["sop_id","title","keywords","version","status"] if c in tbl.columns]
        if not cols:
            return tbl.iloc[0:0]
        mask = tbl[cols].astype(str).apply(lambda s: s.str.lower().str.contains(q, na=False)).any(axis=1)
        return tbl[mask]
    except Exception:
        # list-of-dicts path
        cols = ["sop_id","title","keywords","version","status"]
        return [r for r in tbl if any(q in str(r.get(c, "")).lower() for c in cols)]

# --- optional UI demo (equipment status preview) --------------------------
if getattr(CFG, "run_ui", False):
    try:
        import ipywidgets as W
        import pandas as pd  # UI rendering will prefer pandas if present

        eq_path = CFG.equipment_status_path
        ensure_csv(eq_path, ["equip_id","name","location","status","last_seen","battery","confidence"])

        out = W.Output()
        btn_refresh = W.Button(description="Refresh equipment status", layout=W.Layout(width="250px"))
        btn_add_demo = W.Button(description="Add demo row", layout=W.Layout(width="250px"))

        def _refresh(*_):
            with out:
                out.clear_output()
                try:
                    df = pd.read_csv(eq_path)
                    display(df.head(20))
                except Exception:
                    display(load_csv_df(eq_path)[:20])

        def _add_demo(*_):
            save_csv_row(eq_path, {
                "equip_id": "demo-001",
                "name":     "Portable Monitor",
                "location": "ED-01",
                "status":   "available",
                "last_seen":"", "battery":"", "confidence":""
            })
            _refresh()

        btn_refresh.on_click(_refresh)
        btn_add_demo.on_click(_add_demo)
        _refresh()

        display(W.VBox([W.HBox([btn_refresh, btn_add_demo]), out]))
        print("UI ready → equipment status preview")
    except Exception as e:
        print("UI disabled (ipywidgets/pandas not available):", e)
else:
    print("UI off (CFG.run_ui = False)")

# --- SOP file path (from CFG) ---------------------------------------------
SOP_PATH = CFG.sop_registry_path


UI off (CFG.run_ui = False)


In [18]:
# --- UI helpers (streamlit-optional, single drop-in) ---
# Safely import streamlit; fall back to None in notebook jobs without it
try:
    import streamlit as st
except ModuleNotFoundError:
    st = None

def run_ui_header_controls():
    if st is None:
        # notebook/defaults when Streamlit is unavailable
        return 120, 120
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        vitals_thresh = st.number_input("Vitals overdue (min)", min_value=5, max_value=720, value=120, step=5)
    with c2:
        if st.button("Refresh"):
            st.experimental_rerun()
    return thresh, vitals_thresh

def run_ui_equipment_panel(tracker, overdue_thresh_min: float):
    if st is None:
        print("[UI disabled] Equipment panel skipped."); return
    import pandas as pd
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(overdue_thresh_min))
        st.subheader("Overdue")
        if overdue.empty:
            st.write("None")
        else:
            st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]],
                         use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1:
        sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2:
        loc_from = st.text_input("From", "")
    with mc3:
        loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to)
            st.success(f"Move logged: {sel_id} → {loc_to}")

def run_ui_qr_panel(tracker):
    if st is None:
        print("[UI disabled] QR panel skipped."); return
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt)
            st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None:
                equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload:
                equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try:
                        equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception:
                        equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc)
                st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc:
                st.error("Provide a new location.")
            else:
                st.error("No QR payload detected (image or manual).")

def run_ui_sop_panel():
    if st is None:
        print("[UI disabled] SOP panel skipped."); return
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/")
            st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys()))
            pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]
                st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y])
                        plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0))
                        plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf))
                    st.pyplot(fig)
                except Exception:
                    st.info("Graph display unavailable; showing list instead.")
                    st.write(edges)

def run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh_min: float):
    if st is None:
        print("[UI disabled] Actions & Critic skipped."); return
    import pandas as pd, numpy as np
    st.header("SOPs")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state, "feature_dict"):
        feats = state.feature_dict()
        since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > float(vitals_thresh_min):
                st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > {int(vitals_thresh_min)}")
            else:
                st.success(f"Vitals recently checked: {since_v:.0f} min (≤ {int(vitals_thresh_min)})")
    if st.button("Mark vitals now") and hasattr(state, "touch_now"):
        state.touch_now(pd.Timestamp.utcnow())
        st.success("Vitals timestamp updated.")
    actions = get_actions(state) or []
    if not actions:
        st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    view = pd.DataFrame({
        "id":[a.get("id") for a in actions],
        "label":[a.get("label") for a in actions],
        "p_accept":np.round(p,3),
        "benefit":np.round(benefit,3),
        "burden":np.round(burden,3)
    }).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)

def run_ui_movement_analytics(tracker):
    if st is None:
        print("[UI disabled] Movement analytics skipped."); return
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats()
    per_eq = stats.get("moves_per_equipment"); routes = stats.get("routes")
    if per_eq is None or getattr(per_eq, "empty", True):
        st.info("No movement data yet."); return
    st.subheader("Moves per equipment")
    st.dataframe(per_eq, use_container_width=True, height=240)
    try:
        import matplotlib.pyplot as plt
        fig = plt.figure()
        x = per_eq["equip_id"].astype(str).tolist(); y = per_eq["moves"].tolist()
        plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment")
        st.pyplot(fig)
    except Exception:
        pass
    st.subheader("Top routes")
    if routes is not None:
        st.dataframe(routes, use_container_width=True, height=200)


In [19]:
# --- Wrapper: streamlit-optional (single drop-in) ---
def run_ui(tracker, get_state, get_actions, critic):
    # use global 'st' from helpers; fall back to notebook mode if unavailable
    try:
        st  # defined in helpers cell
    except NameError:
        st = None  # noqa: F841

    if st is None:
        # Notebook fallback (no Streamlit)
        import pandas as pd, numpy as np
        from IPython.display import display
        print("UI — notebook fallback (no Streamlit)")
        overdue_thresh, vitals_thresh = run_ui_header_controls()  # returns defaults (120, 120)

        # Minimal equipment view
        try:
            eq_df = tracker.equipment_status()
            print("\nEquipment (sample):"); display(eq_df.head(20))
        except Exception:
            pass

        # Actions & critic
        state = get_state()
        actions = get_actions(state) or []
        if actions:
            p, benefit, burden = critic.score(state, actions)
            view = pd.DataFrame({
                "id":[a.get("id") for a in actions],
                "label":[a.get("label") for a in actions],
                "p_accept":np.round(p,3),
                "benefit":np.round(benefit,3),
                "burden":np.round(burden,3),
            }).sort_values(["p_accept","benefit"], ascending=[False, False])
            print("\nActions & Critic:"); display(view)
        else:
            print("\nNo actions available.")

        # Movement analytics
        if hasattr(tracker, "movement_stats"):
            stats = tracker.movement_stats()
            per_eq = stats.get("moves_per_equipment"); routes = stats.get("routes")
            if per_eq is not None:
                print("\nMoves per Equipment:"); display(per_eq)
            if routes is not None:
                print("\nTop Routes:"); display(routes)
        print("ui ready")
        return

    # --- Streamlit path ---
    import pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")

    overdue_thresh, vitals_thresh = run_ui_header_controls()
    run_ui_equipment_panel(tracker, overdue_thresh)
    run_ui_qr_panel(tracker)
    run_ui_sop_panel()

    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if getattr(sop_hits, "empty", True):
        st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick_opts = [""] + sop_hits["sop_id"].astype(str).tolist()
        pick = st.selectbox("Open SOP", pick_opts)
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")

    run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh)
    run_ui_movement_analytics(tracker)
    st.caption("ui ready")


In [20]:
def _go(_):
    with out:
        out.clear_output()
        print("Running…")
        try:
            # Minimal smoke: compute Phase-2 bundle and log, then score a dummy row with the loaded MLP
            vitals = {"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}
            labs   = {"d_dimer":780,"d_dimer_unit":"μg/L FEU", "creatinine":1.8, "platelets":95, "albumin":2.8}
            ctx    = {"suspected_infection": True}
            b = phase2_bundle(vitals, labs, ctx)
            print("\n".join(b["one_liners"]))
            if "predict_one" in globals() and "FEAT" in globals():
                row = {f: labs.get(f, vitals.get(f, ctx.get(f, 0.0))) for f in FEAT}
                res = predict_one(row)
                print(f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})")
        except Exception as e:
            print("[ERROR]", e)


In [21]:
def run_icu_constraints(state):
    if not getattr(CFG, "run_pipeline", False): return {}
    mod = _try_import("icu_constraints") or _try_import("modeling_icu_constraints")
    if mod and hasattr(mod,"compute_icu_flags"):
        try: return dict(mod.compute_icu_flags(state))
        except Exception: return {}
    return {}

def mesh_route_actions(state, actions):
    if not getattr(CFG, "run_pipeline", False): return actions
    mod = _try_import("agent_mesh") or _try_import("ed_agent_mesh")
    if mod and hasattr(mod,"route"):
        try: return list(mod.route(state, actions))
        except Exception: return actions
    return actions

def trainer_fit_critic(critic, samples, y):
    if not getattr(CFG, "run_pipeline", False): return critic
    mod = _try_import("trainer") or _try_import("ed_trainer")
    if mod and hasattr(mod,"fit_critic"):
        try: return mod.fit_critic(critic, samples, y)
        except Exception: return critic
    return critic


In [22]:
# Inline UI trigger (compute from current UI inputs, no fixtures)
if getattr(CFG, "run_ui", False):
    try:
        import ipywidgets as W, json
        btn = W.Button(description="Compute from UI inputs + log", button_style="primary")
        out = W.Output()

        def _go(_):
            with out:
                out.clear_output()
                print("Running on current UI inputs…")
                try:
                    # Use what's in the text areas right now
                    vitals = json.loads(vitals_in.value) if 'vitals_in' in globals() else {}
                    labs   = json.loads(labs_in.value)   if 'labs_in'   in globals() else {}
                    ctx    = json.loads(ctx_in.value)    if 'ctx_in'    in globals() else {}

                    bundle = phase2_bundle(vitals, labs, ctx)
                    print("\n".join(bundle["one_liners"]))

                    # ML risk if model is loaded
                    if "predict_one" in globals() and "FEAT" in globals():
                        row = {f: labs.get(f, vitals.get(f, ctx.get(f, 0.0))) for f in FEAT}
                        res = predict_one(row)
                        print(f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})")
                        try:
                            _append_event({"type":"ml_score_ui","keys":list(row.keys()),"res":res})
                        except Exception:
                            pass
                    else:
                        print("(ML bridge not loaded — run the bundle load cell)")

                    _append_event({"type":"phase2_bundle_ui","bundle":bundle})
                    print("\nLogged to:", str(CFG.event_log_path))
                except Exception as e:
                    print("[ERROR]", e)

        btn.on_click(_go)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


UI panel deferred… set CONFIG['RUN_UI']=True.


In [23]:
# 1) Make sure the pieces exist
print("phase2_bundle:", "phase2_bundle" in globals())
print("predict_one  :", "predict_one"   in globals(), "| FEAT:", "FEAT" in globals())

# 2) Click your new button once, then tail the log:
from pathlib import Path, PurePath
p = Path(str(CFG.event_log_path))
print("Log exists:", p.exists(), "→", str(p))

if p.exists():
    tail = p.read_text().strip().splitlines()[-8:]
    for ln in tail:
        print(ln[:200])


phase2_bundle: False
predict_one  : True | FEAT: True
Log exists: False → /kaggle/working/event_log.jsonl


In [24]:
# === sklearn Version Check (no install) ===
import sklearn

current_version = sklearn.__version__
required_version = "1.1.3"

print(f"Current sklearn version: {current_version}")
print(f"Model was trained on:     {required_version}")

if current_version != required_version:
    print("⚠️  Version mismatch: may see warnings when loading the bundle, "
          "but pipeline should still run.")
else:
    print("✅ sklearn version is compatible")


Current sklearn version: 1.2.2
Model was trained on:     1.1.3
⚠️  Version mismatch: may see warnings when loading the bundle, but pipeline should still run.


In [25]:
# ICU availability panel — replaces "actionable vs blocked" with explicit next-bed ETAs.
# No sidecars; all inline; guarded by RUN_UI.
from pathlib import Path
from datetime import datetime, timezone
import json

# Pre-seeded UKE ICUs (public info): names + capacities
UKE_UNITS = [
    {"name": "1A Neurochirurgische Intensivstation", "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1B Neurologische Intensivstation",    "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1C Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1D Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1E Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1F Operative Intensivstation",        "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1G Internistische Intensivstation",   "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiologische Intensivstation",  "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiochirurgische Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H2b Intensivstation Gefäß- und Herzmedizin", "capacity": 8, "occupied": 8, "discharge_eta_minutes": []},
]

def _format_eta(mins: int) -> str:
    if mins is None: return "unknown"
    if mins <= 0: return "now"
    h, m = divmod(int(mins), 60)
    return f"{m} min" if h == 0 else (f"{h} hr" if m == 0 else f"{h} hr {m} min")

def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"):
                return js
        except Exception:
            pass
    # default to UKE units when nothing saved
    return {"units": list(UKE_UNITS), "timestamp": datetime.now(timezone.utc).isoformat()}

def _save_icu_status(js, path="/mnt/data/icu_status.json"):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js, ensure_ascii=False, indent=2))

def _compute_next_bed_eta(unit):
    cap = int(unit.get("capacity", 0) or 0)
    occ = int(unit.get("occupied", 0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None

if getattr(CFG, "run_ui", False):
    try:
        import ipywidgets as W
        import pandas as pd

        state = _load_icu_status()
        units = state["units"]

        # UI widgets
        dd = W.Dropdown(options=[u.get("name","(unnamed)") for u in units] or ["(add a unit)"], description="Unit")
        name = W.Text(description="Name", placeholder="ICU-North")
        cap  = W.IntText(description="Capacity", value=12)
        occ  = W.IntText(description="Occupied", value=12)
        eta  = W.Text(description="ETAs (min)", placeholder="e.g. 30, 90, 180")

        add_btn = W.Button(description="Add/Update unit")
        calc_btn = W.Button(description="Estimate ETAs")
        save_btn = W.Button(description="Save status")
        out = W.Output()

        def _refresh_dropdown():
            dd.options = [u.get("name","(unnamed)") for u in units] or ["(add a unit)"]

        def _load_into_form(idx=0):
            if not units:
                name.value=""; cap.value=12; occ.value=12; eta.value=""; return
            u = units[idx]
            name.value = str(u.get("name",""))
            cap.value = int(u.get("capacity", 0) or 0)
            occ.value = int(u.get("occupied", 0) or 0)
            seq = u.get("discharge_eta_minutes") or []
            eta.value = ", ".join(str(int(x)) for x in seq)

        def _parse_eta(txt: str):
            out = []
            for chunk in txt.split(","):
                chunk = chunk.strip()
                if chunk:
                    try: out.append(int(float(chunk)))
                    except: pass
            return out

        def on_dd_change(change):
            if change["name"]=="value" and units:
                _load_into_form(dd.options.index(change["new"]))
        dd.observe(on_dd_change)

        def on_add(_):
            # no 'nonlocal' needed: we mutate the existing list
            u = {
                "name": name.value.strip() or f"ICU-{len(units)+1}",
                "capacity": int(cap.value or 0),
                "occupied": int(occ.value or 0),
                "discharge_eta_minutes": _parse_eta(eta.value),
            }
            names = [x.get("name","") for x in units]
            if u["name"] in names:
                units[names.index(u["name"])] = u
            else:
                units.append(u)
            _refresh_dropdown()
            dd.value = u["name"]
            with out:
                print(f"Saved unit '{u['name']}'")

        def on_calc(_):
            rows = []
            for u in units:
                eta_min = _compute_next_bed_eta(u)
                rows.append({
                    "ICU": u.get("name",""),
                    "capacity": int(u.get("capacity",0) or 0),
                    "occupied": int(u.get("occupied",0) or 0),
                    "next_bed_in": _format_eta(eta_min),
                })
            df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ICU","capacity","occupied","next_bed_in"])
            with out:
                out.clear_output()
                if df.empty:
                    print("No units yet. Add a unit above.")
                else:
                    display(df.style.hide(axis='index'))
                    # Natural-language earliest
                    mins = [(r["ICU"], _compute_next_bed_eta(u)) for r,u in zip(rows, units)]
                    mins = [(n,m) for n,m in mins if m is not None]
                    if mins:
                        name_min, m = sorted(mins, key=lambda x: x[1])[0]
                        print(f"\nNext bed available on {name_min} in {_format_eta(m)}")
                    else:
                        print("\nNext bed availability: unknown (provide ETAs or reduce occupied < capacity).")

        def on_save(_):
            state["units"] = units
            state["timestamp"] = datetime.now(timezone.utc).isoformat()
            _save_icu_status(state)
            with out:
                print("Saved to /mnt/data/icu_status.json")

        add_btn.on_click(on_add)
        calc_btn.on_click(on_calc)
        save_btn.on_click(on_save)

        # Initial load
        _refresh_dropdown()
        if units:
            dd.value = dd.options[0]
            _load_into_form(0)

        display(W.VBox([
            W.HTML("<b>ICU next-bed availability</b>"),
            dd,
            W.HBox([name, cap, occ]),
            eta,
            W.HBox([add_btn, calc_btn, save_btn]),
            out
        ]))

    except Exception as e:
        print("ICU UI unavailable:", e)
else:
    print("ICU UI deferred… set CONFIG['RUN_UI']=True.")


ICU UI deferred… set CONFIG['RUN_UI']=True.


In [26]:
# === Synth helpers (timestamps + case generator) ===
# Pure helpers. No side effects. Uses only fields consumed by gate_engine/ops_intel/troponin_policy.

import random, json
from datetime import datetime, timezone

def _ts() -> str:
    return datetime.now(timezone.utc).isoformat()

def synth_patient(seed=None) -> dict:
    """
    Generate one realistic ED case.
    SBP-centric vitals, optional VBG, abdomen guardrails, and troponin timing for due/overdue.
    """
    rng = random.Random(seed if seed is not None else random.randint(0, 2**31-1))
    case = {}

    # Triage & vitals (SBP, not MAP)
    case["Triage"] = rng.choices(["Green","Yellow","Orange","Red"], weights=[45,30,20,5], k=1)[0]
    case["SBP"]    = max(70, int(rng.gauss(115, 20)))              # mmHg
    case["HR"]     = max(40, int(rng.gauss(92, 20)))
    case["SpO2"]   = max(70, int(rng.gauss(95, 4)))
    case["GCS"]    = rng.choices([15, 14, 13], weights=[90,7,3])[0]

    # Chief complaint
    cc = rng.choices(
        ["Brustschmerz","Bauchschmerz","Dyspnoe","Synkope","Fieber","Schwindel"],
        weights=[18,22,18,10,20,12], k=1
    )[0]
    case["Leitsymptom"] = cc

    # VBG in ~40%: mostly normal, but inject tails to exercise gates/alerts
    if rng.random() < 0.4:
        case["vbg_pH"]        = round(rng.gauss(7.40, 0.06), 2)
        case["vbg_pco2_mmHg"] = round(max(30, rng.gauss(42, 12)), 1)
        case["vbg_HCO3"]      = round(max(8, rng.gauss(24, 5)), 1)
        case["vbg_lactate"]   = round(max(0.5, rng.gauss(1.8, 0.9)), 1)
        case["vbg_Na"]        = round(rng.gauss(139, 6), 1)
        case["vbg_Cl"]        = round(rng.gauss(103, 6), 1)
        case["vbg_K"]         = round(rng.gauss(4.1, 0.7), 1)

        # Rare criticals (aligned with your ops_intel thresholds)
        if rng.random() < 0.05: case["vbg_pH"]      = 7.56   # alkalosis critical (>7.55)
        if rng.random() < 0.05: case["vbg_K"]       = 6.6    # hyperK critical (>=6.5)
        if rng.random() < 0.05: case["vbg_lactate"] = 4.3    # lactate critical (>=4.0)
        if rng.random() < 0.03: case["vbg_Na"]      = 124    # hyponatraemia critical (<=125)

    # Abdomen guardrails (force unmet requirements to trigger gates)
    if "bauch" in cc.lower() or "abd" in cc.lower():
        case["US_done"] = False; case["CT_done"] = False
        case["labs_cycles_done"] = 1

    # Troponin: enable due/overdue logic; we’re not doing assay inference here
    if cc == "Brustschmerz" and rng.random() < 0.7:
        case["troponin0_ngL"]  = rng.choice([8, 10, 14, 20, 55])  # includes potential rule-in t0
        case["since_trop1_min"]= rng.choice([30, 65, 80])         # to trigger DUE/OVERDUE
        case["admit_decision"] = rng.random() < 0.5

    return case


In [27]:
# === Gate-only stream emitter (no MLP, writes JSONL) ===
# Pure function; no side effects unless you call it.

from collections import Counter
from pathlib import Path
from typing import Dict, Any

def emit_events(n: int = 50, seed: int = 42, sleep_s: float = 0.0,
                CONFIG: Dict[str, Any] = None, EVENT_LOG_PATH: str | None = None):
    assert CONFIG is not None, "CONFIG required"
    log_path = EVENT_LOG_PATH or str(CFG.event_log_path)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    Path(log_path).touch(exist_ok=True)

    gate_counts, prio_hist, actions = Counter(), Counter(), Counter()

    for i in range(n):
        fd   = synth_patient(seed + i)
        pack = gate_engine(fd, CONFIG)  # uses your troponin_policy + ops_intel

        for g in pack.get("gates", []): gate_counts[g] += 1
        prio_hist[pack.get("priority", 0)] += 1
        if pack.get("next_action"): actions[pack["next_action"]] += 1

        rec = {
            "ts": _ts(),
            "case_id": f"synth-{seed}-{i}",
            "fd": fd,
            "gates": pack.get("gates", []),
            "priority": pack.get("priority", 0),
            "next_action": pack.get("next_action", ""),
            "ttl": pack.get("ttl", {}),
            "explain": pack.get("explain", []),
        }
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if sleep_s:  # optional pacing
            import time; time.sleep(float(sleep_s))

    # Optional tabular summary (pandas if available)
    try:
        import pandas as pd
        df = (pd.DataFrame({"gate": list(gate_counts.keys()), "count": list(gate_counts.values())})
                .sort_values("count", ascending=False).reset_index(drop=True))
        print("Gate frequency (top 10):"); display(df.head(10))
    except Exception:
        print("Gate frequency (top 10):", gate_counts.most_common(10))
    print("Priority histogram:", dict(prio_hist))
    print("Top actions:", actions.most_common(5))

    return {"gate_counts": gate_counts, "prio_hist": prio_hist, "actions": actions, "log_path": log_path}


In [28]:
# === ICU capacity gate-pack (tiny & pure; reuses ICU_PREFS if present) ===
from typing import Dict, Any, List

def _best_relevant_icu_capacity(fd: Dict[str, Any], ICU_PREFS: Dict[str, List[str]]) -> float:
    cond = fd.get("suspected_condition")
    keys = ICU_PREFS.get(cond, [])
    vals = []
    for k in keys:
        try: vals.append(float(fd.get(k, 0.0)))
        except Exception: pass
    return max(vals) if vals else 0.0

def capacity_gatepack(fd: Dict[str, Any], CONFIG: Dict[str, Any]) -> Dict[str, Any]:
    # Prefer your panel's ICU_PREFS if it exists; else fallback
    ICU_PREFS = CONFIG.get("ICU_PREFS", globals().get("ICU_PREFS", {
        "cardiac":         ["cap_cardio_icu","cap_cardio_surgery_icu","cap_vascular_cardiac_icu"],
        "respiratory":     ["cap_internal_medicine_icu","cap_interdis_stage1","cap_interdis_stage2","cap_interdis_stage3"],
        "neurological":    ["cap_neurological_icu","cap_neurochirurgical_icu","cap_interdis_stage1"],
        "infection":       ["cap_internal_medicine_icu","cap_interdis_stage2"],
        "trauma":          ["cap_surgical_icu","cap_interdis_stage3"],
        "oncology":        ["cap_internal_medicine_icu","cap_interdis_stage2"],
        "gastrointestinal":["cap_surgical_icu","cap_internal_medicine_icu"],
    }))
    TH_OK    = float(CONFIG.get("TH_ICU_CAP_OK",    0.25))  # ≥ OK
    TH_TIGHT = float(CONFIG.get("TH_ICU_CAP_TIGHT", 0.15))  # < tight
    SLA_ESC  = int(CONFIG.get("SLA_ADMIT_ESCALATE_MIN", 60))

    triage_txt = str(fd.get("Triage","")).title()
    urgent = (int(fd.get("ems_triage_code", 3)) <= 2) or (triage_txt in ("Red","Orange"))

    cap_rel    = _best_relevant_icu_capacity(fd, ICU_PREFS)  # 0..1
    bottleneck = 1.0 - cap_rel

    gates: List[str] = []; explain: List[str] = []; ttl: Dict[str,int] = {}
    prio, action = 0, ""

    if cap_rel <= 0.0:
        gates += ["ICU_BLOCKED"]; explain += ["Relevant ICU capacity = 0%"]
        prio = max(prio, 5); action = action or "No ICU capacity — board in ED, activate bed manager now."
    elif cap_rel < TH_TIGHT:
        gates += ["ICU_TIGHT"]; explain += [f"Relevant ICU capacity < {int(TH_TIGHT*100)}%"]
        prio = max(prio, 5 if urgent else 4); action = action or "Tight ICU capacity — escalate to bed manager."
    elif cap_rel >= TH_OK:
        gates += ["ICU_OK"]; explain += [f"Relevant ICU capacity ≥ {int(TH_OK*100)}%"]
        prio = max(prio, 3 if urgent else 2); action = action or "Proceed with ICU handoff planning."

    if bottleneck >= 0.70:
        gates += ["ED_BOARDING_RISK"]; explain += ["High boarding risk from ICU bottleneck"]
        prio = max(prio, 4)

    if urgent and cap_rel >= TH_OK:
        gates += ["ROUTE_PLAN_EMS_TO_ICU"]; explain += ["Urgent + capacity OK → EMS→ICU"]
    else:
        gates += ["ROUTE_PLAN_ED_TO_ICU"];  explain += ["Default or constrained → EMS→ED→ICU"]

    if any(g in ("ICU_BLOCKED","ICU_TIGHT") for g in gates):
        for g in ("ALERT_NURSE","ALERT_DOC"):
            if g not in gates: gates.append(g); ttl[g] = 300
        if ("ICU_BLOCKED" in gates) or (urgent and "ICU_TIGHT" in gates):
            if "ALERT_ATTENDING" not in gates: gates.append("ALERT_ATTENDING"); ttl["ALERT_ATTENDING"] = 300

    if str(fd.get("admit_decision","")).upper() == "ICU" and any(g in ("ICU_BLOCKED","ICU_TIGHT") for g in gates):
        gates.append("TRANSFER_BLOCK_CAPACITY"); explain.append(f"Recheck capacity q{SLA_ESC}min until cleared")
        ttl["TRANSFER_BLOCK_CAPACITY"] = SLA_ESC * 60

    # de-dup preserve order
    seen=set(); gates=[g for g in gates if (g not in seen and not seen.add(g))]
    return {"gates": gates, "priority": prio, "next_action": action, "explain": explain, "ttl": ttl}


In [29]:
# ------------------------------
# Helpers (self-contained)
# ------------------------------
from __future__ import annotations

# stdlib
from pathlib import Path
import os
import re

# third-party
import numpy as np
import pandas as pd

# public API of this cell
__all__ = ["_try_read_csv", "_to_dt", "_safe_num", "tokenize"]

# --- lightweight tokenizer helpers ---
_PUNCT_RE = re.compile(r"[\W_]+", re.UNICODE)
_STOP = set("""
a an and the of for to with in on at by from as is are was were be been being this that these those it its into out up
down over under about above below after before not no nor or but so such very more most less least few many much other
another own same also just can could should would will may might must than then there here when where why how who whom
which what whose via per without within across between during each either neither both all any some one two three four
five six seven eight nine ten
""".split())

def tokenize(text: object) -> list[str]:
    """Tiny tokenizer for chief complaints; lowercases, strips punctuation, drops stopwords/digits."""
    if not isinstance(text, str):
        return []
    text = _PUNCT_RE.sub(" ", text.lower())
    return [t for t in text.split() if t and t not in _STOP and not t.isdigit()]

# --- IO + coercion helpers ---
def _try_read_csv(path: Path | str) -> pd.DataFrame:
    """Read CSV (or CSV.GZ). Accepts full filename or stem without suffix."""
    p = Path(path)
    # exact path
    if p.exists():
        return pd.read_csv(p, compression="infer", low_memory=False)
    # same name with .gz
    if p.suffix:
        gz = p.with_suffix(p.suffix + ".gz")
        if gz.exists():
            return pd.read_csv(gz, compression="infer", low_memory=False)
    else:
        # try stem.csv then stem.csv.gz
        for cand in (p.with_suffix(".csv"), p.with_suffix(".csv.gz")):
            if cand.exists():
                return pd.read_csv(cand, compression="infer", low_memory=False)
    raise FileNotFoundError(f"Could not find CSV for {p} (tried .csv and .csv.gz)")

def _to_dt(s: pd.Series) -> pd.Series:
    """Coerce to datetime with NaT on failure."""
    return pd.to_datetime(s, errors="coerce")

def _safe_num(s: pd.Series) -> pd.Series:
    """Coerce to numeric with NaN on failure."""
    return pd.to_numeric(s, errors="coerce")


In [30]:
# === Data loader (MIMIC demo; robust; SPC-ready) ===
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
import numpy as np
import os

@dataclass(frozen=True)
class DataPaths:
    root: Path = Path("/kaggle/input/mimic-iv-demo-v2-2")
    edstays: Path = None
    diagnosis: Path = None
    medrecon: Path = None

    def __post_init__(self):
        object.__setattr__(self, "edstays",  self.edstays  or (self.root / "edstays.csv"))
        object.__setattr__(self, "diagnosis", self.diagnosis or (self.root / "diagnosis.csv"))
        object.__setattr__(self, "medrecon",  self.medrecon  or (self.root / "medrecon.csv"))

def _read_csv_safe(path: Path, usecols=None, parse_dates=None) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, usecols=usecols)
    except Exception:
        df = pd.read_csv(path)
    # parse dates if present
    parse_dates = parse_dates or []
    for col in parse_dates:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    return df

def load_mimic_demo(paths: DataPaths = DataPaths()) -> dict:
    """Load core tables with safe parsing; returns dict with 'ed', 'dx', 'meds'."""
    ed = _read_csv_safe(paths.edstays, parse_dates=["intime","outtime","arrival_time","depart_time"])
    dx = _read_csv_safe(paths.diagnosis)
    meds = _read_csv_safe(paths.medrecon)
    # normalize common column names
    if "stay_id" in ed.columns:
        ed.rename(columns={"stay_id":"ed_stay_id"}, inplace=True)
    for c_old, c_new in [("admit_decision","admit_decision"),
                         ("disposition","disposition"),
                         ("triage","triage"),
                         ("Triage","triage")]:
        if c_old in ed.columns and c_new != c_old:
            ed.rename(columns={c_old: c_new}, inplace=True)
    # ensure time columns exist
    for c in ["intime","outtime","arrival_time","depart_time"]:
        if c not in ed.columns:
            ed[c] = pd.NaT
    return {"ed": ed, "dx": dx, "meds": meds}

DATA = load_mimic_demo()
print("Loaded:", {k: v.shape for k, v in DATA.items()})


Loaded: {'ed': (222, 11), 'dx': (545, 6), 'meds': (2764, 9)}


In [31]:
# === PATCH: robust feature engineering + daily aggregate ===
import pandas as pd
import numpy as np

def _first_non_null(series, fallback=np.nan):
    for v in series:
        if pd.notna(v):
            return v
    return fallback

def engineer_ed_features(ed: pd.DataFrame) -> pd.DataFrame:
    """Per-visit engineered features (robust to missing/mixed dtypes)."""
    df = ed.copy()

    # pick best available start/end timestamps, then coerce to UTC-aware datetimes
    start_candidates = [c for c in ["intime","arrival_time"] if c in df.columns]
    end_candidates   = [c for c in ["outtime","depart_time"] if c in df.columns]

    df["ts_start"] = (
        df[start_candidates].apply(lambda r: _first_non_null(r, pd.NaT), axis=1)
        if start_candidates else pd.NaT
    )
    df["ts_end"] = (
        df[end_candidates].apply(lambda r: _first_non_null(r, pd.NaT), axis=1)
        if end_candidates else pd.NaT
    )

    # force to datetime with UTC (handles object dtype / mixed columns safely)
    df["ts_start"] = pd.to_datetime(df["ts_start"], errors="coerce", utc=True)
    df["ts_end"]   = pd.to_datetime(df["ts_end"],   errors="coerce", utc=True)

    # LOS in hours
    df["los_h"] = (df["ts_end"] - df["ts_start"]).dt.total_seconds() / 3600.0

    # normalize strings if present
    if "triage" in df.columns:
        df["triage_norm"] = df["triage"].astype(str).str.strip().str.title()
    if "admit_decision" in df.columns:
        df["admit_decision_norm"] = df["admit_decision"].astype(str).str.strip().str.upper()

    # day stamp (only where ts_start is valid)
    df["day"] = df["ts_start"].dt.floor("D")

    return df

def spc_aggregate_daily(ed_feat: pd.DataFrame) -> pd.DataFrame:
    """Daily aggregates with guaranteed 'day' column and safe empty handling."""
    cols = ["day","n_arrivals","n_icudadmit","los_h_mean","los_h_p90"]

    if "day" not in ed_feat.columns:
        return pd.DataFrame(columns=cols)

    df = ed_feat.dropna(subset=["day"]).copy()
    if df.empty:
        return pd.DataFrame(columns=cols)

    g = df.groupby("day")

    n_arrivals = g.size().rename("n_arrivals")
    if "admit_decision_norm" in df.columns:
        n_icudadmit = g["admit_decision_norm"].apply(lambda s: (s == "ICU").sum()).rename("n_icudadmit")
    else:
        n_icudadmit = pd.Series(index=n_arrivals.index, dtype=int, name="n_icudadmit")

    los_h_mean = g["los_h"].mean().rename("los_h_mean")
    los_h_p90  = g["los_h"].quantile(0.90).rename("los_h_p90")

    agg = pd.concat([n_arrivals, n_icudadmit, los_h_mean, los_h_p90], axis=1).reset_index()
    # ensure 'day' exists even if concat produced no columns (defensive)
    if "day" not in agg.columns:
        agg["day"] = pd.NaT
        agg = agg[cols]
    else:
        agg = agg.sort_values("day")
        # add any missing columns
        for c in cols:
            if c not in agg.columns:
                agg[c] = np.nan
        agg = agg[cols]

    return agg


In [32]:
# Consolidated & statistically-rigorous SPCProcessControl
# Replaces later duplicate; keeps earlier placement. Default lookback_weeks=12 as requested.
import math
import warnings
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

import numpy as np
import pandas as pd

def _lazy_has_scipy():
    try:
        import scipy.stats  # noqa: F401
        return True
    except Exception:
        return False

@dataclass
class BaselineStats:
    mean: float
    std: float
    lcl: float
    ucl: float
    cl: float
    chart_type: str
    subgroup_size: Optional[int] = None
    normality_ok: Optional[bool] = None
    details: Optional[dict] = None

class SPCProcessControl:
    def __init__(self, sigma: float = 3.0, min_samples: int = 30, lookback_weeks: int = 12):
        if sigma <= 0:
            raise ValueError("sigma must be > 0")
        if min_samples < 1:
            raise ValueError("min_samples must be >= 1")
        if lookback_weeks < 1:
            raise ValueError("lookback_weeks must be >= 1")
        self.sigma = float(sigma)
        self.min_samples = int(min_samples)
        self.lookback_weeks = int(lookback_weeks)
        self._baseline: Optional[BaselineStats] = None

    # --------- Preprocessing ---------
    @staticmethod
    def preprocess_series(ts: pd.Series, freq: str = "D",
                          cap_lower_q: float = 0.01, cap_upper_q: float = 0.99,
                          log_transform_if_skewed: bool = True) -> pd.Series:
        """
        Robust preprocessing:
        - Enforce monotonic DateTimeIndex (if possible).
        - Interpolate missing timestamps and values using time-based interpolation.
        - Cap extreme outliers at quantiles (winsorize).
        - Log1p-transform highly skewed nonnegative counts.
        """
        if not isinstance(ts, pd.Series):
            raise TypeError("ts must be a pandas Series")
        if not isinstance(ts.index, pd.DatetimeIndex):
            try:
                ts.index = pd.to_datetime(ts.index)
            except Exception:
                # Keep as-is; interpolation will be skipped
                pass
        ts = ts.sort_index()

        # Create full time index at given frequency if DatetimeIndex
        if isinstance(ts.index, pd.DatetimeIndex):
            full_idx = pd.date_range(ts.index.min(), ts.index.max(), freq=freq)
            ts = ts.reindex(full_idx)
            # Interpolate missing values
            ts = ts.interpolate(method="time").ffill().bfill()
        else:
            ts = ts.interpolate(method="linear").ffill().bfill()

        # Cap extremes
        lo = ts.quantile(cap_lower_q)
        hi = ts.quantile(cap_upper_q)
        ts = ts.clip(lower=lo, upper=hi)

        # Log-transform if skewed and nonnegative
        if log_transform_if_skewed and (ts.min() >= 0):
            skew = ((ts - ts.mean())**3).mean() / (ts.std(ddof=0)**3 + 1e-12)
            if abs(skew) > 1.0:
                ts = np.log1p(ts)
                ts.attrs["log_transformed"] = True
            else:
                ts.attrs["log_transformed"] = False
        return ts

    # --------- Normality Test ---------
    @staticmethod
    def normality_ok(sample: pd.Series, alpha: float = 0.05) -> Tuple[bool, Dict]:
        """
        Prefer scipy.stats for Shapiro/DAgostino. If unavailable, use heuristic on skew & kurtosis.
        Returns (ok, details)
        """
        s = pd.Series(sample).dropna()
        details = {}
        ok = True
        try:
            if _lazy_has_scipy():
                import scipy.stats as st  # type: ignore
                if len(s) >= 8 and len(s) <= 5000:
                    stat, p = st.shapiro(s.sample(n=min(5000, len(s)), random_state=0))  # Shapiro is robust for small n
                    details["test"] = "shapiro"
                    details["stat"] = float(stat)
                    details["p"] = float(p)
                    ok = bool(p >= alpha)
                else:
                    stat, p = st.normaltest(s.values)  # D'Agostino-Pearson
                    details["test"] = "normaltest"
                    details["stat"] = float(stat)
                    details["p"] = float(p)
                    ok = bool(p >= alpha)
            else:
                # Heuristic fallback
                skew = s.skew()
                kurt_excess = s.kurt()  # excess if pandas uses Fisher; document approximate
                details["test"] = "heuristic_skew_kurt"
                details["skew"] = float(skew)
                details["kurt_excess"] = float(kurt_excess)
                ok = (abs(skew) < 1.0) and (abs(kurt_excess) < 1.0)
        except Exception as e:
            details["error"] = repr(e)
            # fallback to heuristic
            skew = s.skew()
            kurt_excess = s.kurt()
            details["test"] = "heuristic_skew_kurt_fallback"
            details["skew"] = float(skew)
            details["kurt_excess"] = float(kurt_excess)
            ok = (abs(skew) < 1.0) and (abs(kurt_excess) < 1.0)
        return ok, details

    # --------- Fit Baseline ---------
    def fit(self, series: pd.Series, subgroup_size: Optional[int] = None,
            use_moving_range: bool = True, freq: str = "D") -> BaselineStats:
        """
        Build baseline on the most recent lookback_weeks of data.
        If subgroup_size >= 2 -> Xbar-R; else Individuals (I) chart with Moving Range (MR).
        """
        s = pd.Series(series).dropna()
        if len(s) < max(self.min_samples, 5):
            raise ValueError(f"Not enough samples to fit: have {len(s)}, need >= {self.min_samples}")

        # Restrict to lookback window if datetime index
        if isinstance(s.index, pd.DatetimeIndex):
            cutoff = s.index.max() - pd.Timedelta(weeks=self.lookback_weeks)
            s_lb = s[s.index > cutoff]
            if len(s_lb) >= self.min_samples:
                s = s_lb

        s = self.preprocess_series(s, freq=freq)

        norm_ok, details = self.normality_ok(s)

        # Choose chart
        if subgroup_size and subgroup_size >= 2:
            # Xbar-R chart
            # Split into subgroups in order
            n = int(subgroup_size)
            groups = [s.values[i:i+n] for i in range(0, len(s), n)]
            groups = [g for g in groups if len(g) == n]
            if len(groups) < 2:
                raise ValueError("Not enough full subgroups for Xbar-R chart.")
            xbars = np.array([np.mean(g) for g in groups])
            ranges = np.array([np.max(g) - np.min(g) for g in groups])
            xbarbar = float(np.mean(xbars))
            rbar = float(np.mean(ranges))
            # Constants for Xbar-R (n up to 25). Use d2 to estimate sigma via rbar/d2.
            # Minimal table for d2 and A2 where n=2..10; extend if needed.
            d2_table = {2:1.128,3:1.693,4:2.059,5:2.326,6:2.534,7:2.704,8:2.847,9:2.970,10:3.078}
            A2_table = {2:1.880,3:1.023,4:0.729,5:0.577,6:0.483,7:0.419,8:0.373,9:0.337,10:0.308}
            d2 = d2_table.get(n, None)
            A2 = A2_table.get(n, None)
            if d2 is None or A2 is None:
                # approximate sigma via MR if d2 unknown
                d2 = 1.128  # fallback (n=2) conservative
                A2 = 1.880
            sigma_est = rbar / d2 if d2 > 0 else float(np.std(s.values, ddof=1))
            ucl = xbarbar + self.sigma * sigma_est / math.sqrt(n)
            lcl = xbarbar - self.sigma * sigma_est / math.sqrt(n)
            self._baseline = BaselineStats(mean=xbarbar, std=sigma_est, lcl=lcl, ucl=ucl, cl=xbarbar,
                                           chart_type="Xbar-R", subgroup_size=n, normality_ok=norm_ok, details=details)
        else:
            # Individuals chart with Moving Range for sigma estimation
            x = s.values.astype(float)
            mr = np.abs(np.diff(x))
            mr_bar = float(np.mean(mr)) if len(mr) > 0 else float(np.std(x, ddof=1))*1.128
            # d2 for MR with n=2 is 1.128
            sigma_est = mr_bar / 1.128 if mr_bar > 0 else float(np.std(x, ddof=1))
            mean = float(np.mean(x))
            ucl = mean + self.sigma * sigma_est
            lcl = mean - self.sigma * sigma_est
            self._baseline = BaselineStats(mean=mean, std=sigma_est, lcl=lcl, ucl=ucl, cl=mean,
                                           chart_type="I-MR" if use_moving_range else "Individuals",
                                           subgroup_size=1, normality_ok=norm_ok, details=details)
        return self._baseline

    # --------- Scoring ---------
    def calculate_control_score(self, value: float, baseline: Optional[BaselineStats] = None) -> float:
        """
        Binary significance score using z-score against baseline std and sigma threshold.
        """
        bl = baseline or self._baseline
        if bl is None or bl.std <= 0:
            return 0.0
        z = abs(float(value) - bl.mean) / bl.std
        return 1.0 if z > self.sigma else 0.0

    def p_value_score(self, value: float, baseline: Optional[BaselineStats] = None) -> float:
        """
        Alternative: convert z to (two-sided) p-value; score closer to 1 for rarer events.
        """
        bl = baseline or self._baseline
        if bl is None or bl.std <= 0:
            return 0.0
        z = abs(float(value) - bl.mean) / bl.std
        try:
            if _lazy_has_scipy():
                import scipy.stats as st
                p = 2*(1 - st.norm.cdf(z))
            else:
                # error function approximation
                p = 2*(1 - 0.5*(1+math.erf(z/math.sqrt(2))))
        except Exception:
            p = math.exp(-0.5*z*z)  # crude tail approximation
        # Map smaller p to higher score: score = 1 - min(1, p/alpha) with alpha ~ 0.05
        alpha = 0.05
        return float(max(0.0, 1.0 - min(1.0, p/alpha)))

    def robust_z_score(self, value: float, series_window: Optional[pd.Series] = None) -> float:
        """
        Alternative: robust z via MAD for non-normal data.
        """
        s = pd.Series(series_window).dropna() if series_window is not None else None
        if s is None or len(s) < 5:
            bl = self._baseline
            if bl is None or bl.std <= 0:
                return 0.0
            return abs(float(value) - bl.mean) / bl.std
        med = float(s.median())
        mad = float((s - med).abs().median())
        if mad <= 0:
            return 0.0
        # Consistency constant for normal: 1.4826
        return abs(float(value) - med) / (1.4826 * mad)

    def run_rules_score(self, series_tail: pd.Series, baseline: Optional[BaselineStats] = None) -> float:
        """
        Western Electric style rules aggregated into a [0,1] score.
        """
        bl = baseline or self._baseline
        if bl is None or bl.std <= 0:
            return 0.0
        x = pd.Series(series_tail).dropna().values.astype(float)
        if len(x) < 8:
            return 0.0
        cl = bl.cl
        s = bl.std
        # Rules: 1 point beyond 3σ; 2 of 3 beyond 2σ on same side; 4 of 5 beyond 1σ on same side; 8 in a row on same side
        score = 0.0
        # Rule 1
        if (x[-1] > cl + 3*s) or (x[-1] < cl - 3*s):
            score += 0.4
        # Rule 2
        last3 = x[-3:]
        if (np.sum(last3 > cl + 2*s) >= 2) or (np.sum(last3 < cl - 2*s) >= 2):
            score += 0.25
        # Rule 3
        last5 = x[-5:]
        if (np.sum(last5 > cl + 1*s) >= 4) or (np.sum(last5 < cl - 1*s) >= 4):
            score += 0.2
        # Rule 4
        side = np.sign(x - cl)
        run_len = 1
        for i in range(1, len(side)):
            if side[-i] != 0 and side[-i] == side[-i-1]:
                run_len += 1
            else:
                break
        if run_len >= 8:
            score += 0.15
        return float(min(1.0, score))

    # --------- EWMA & CUSUM helpers (for reference use) ---------
    @staticmethod
    def ewma(series: pd.Series, lambda_: float = 0.2) -> pd.Series:
        """Exponentially Weighted Moving Average."""
        s = pd.Series(series).astype(float)
        return s.ewm(alpha=lambda_, adjust=False).mean()

    @staticmethod
    def cusum(series: pd.Series, k: float = 0.5, h: float = 5.0) -> pd.DataFrame:
        """
        Two-sided CUSUM. k is the reference value (in sigma units), h is the decision interval.
        Returns DataFrame with Cplus, Cminus, signals.
        """
        x = pd.Series(series).astype(float)
        mu = x.mean()
        sigma = x.std(ddof=1) + 1e-12
        k_abs = k * sigma
        cp = [0.0]
        cm = [0.0]
        sig = [False]
        for i in range(1, len(x)):
            cp.append(max(0.0, cp[-1] + (x.iloc[i] - mu - k_abs)))
            cm.append(max(0.0, cm[-1] + (mu - x.iloc[i] - k_abs)))
            sig.append(cp[-1] > h * sigma or cm[-1] > h * sigma)
        return pd.DataFrame({"Cplus": cp, "Cminus": cm, "signal": sig}, index=x.index)



In [33]:
# === Calc utils (shared, model-free) ===
from typing import Dict, Any, Optional, List, Tuple
import re, math
from datetime import datetime

def _pick(d: Dict[str, Any], names: List[str], cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try:
                return cast(d[n])
            except Exception:
                try:
                    return cast(str(d[n]).replace(",", "."))
                except Exception:
                    pass
    return default

def _bool(v) -> bool:
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"1","true","yes","y","ja","on","oui"}

def _safe_round(x, nd=1):
    try:
        return round(float(x), nd)
    except Exception:
        return x
# === Phase-2 calculators A (core scores; model-free) ===
from typing import Dict, Any


In [34]:
# === Phase-2 calculators A (core scores; model-free) ===
from typing import Dict, Any

def calc_qsofa(v: Dict[str,Any]):
    rr=_pick(v,["rr","resp_rate"]); sbp=_pick(v,["sbp","systolic"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v: Dict[str,Any]):
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else 0)
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v: Dict[str,Any], labs: Dict[str,Any]):
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2:
        try: pf=float(pao2)/float(fio2)
        except Exception: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)  # presence-guarded (MAP not required)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

def calc_sirs(p: Dict[str,Any]):
    crit = {
        "temp>38/<36": int(((_pick(p,["temp"],float) or 37)>38) or ((_pick(p,["temp"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","pulse"],float) or 0) > 90),
        "rr>20/paco2<32": int(((_pick(p,["rr"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12/<4/bands>10%": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values()))}

def sepsis3_screen(v: Dict[str,Any], labs: Dict[str,Any], ctx: Dict[str,Any], sofa_min: Dict[str,Any], qsofa: Dict[str,Any]):
    sus = _bool(ctx.get("suspected_infection")); on_pressors=_bool(ctx.get("vasopressors"))
    mapv=_pick(v,["map"]); lact=_pick(labs,["lactate"])
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    return {"name":"SEPSIS3","sepsis_flag": bool(sus and sofa_min["score"]>=2), "septic_shock": septic_shock}


In [35]:
# === Phase-2 calculators B (PE/VTE, GI/hepatic, corrections + lab assess) ===
from typing import Dict, Any, List, Optional, Tuple

def calc_wells_pe(p: Dict[str,Any]):
    pts=0.0
    pts+=3.0 if _bool(p.get("dvt_signs")) else 0.0
    pts+=3.0 if _bool(p.get("pe_most_likely")) else 0.0
    pts+=1.5 if ((_pick(p,["hr","pulse"],float) or 0)>100) else 0.0
    pts+=1.5 if (_bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w"))) else 0.0
    pts+=1.5 if _bool(p.get("prev_vte")) else 0.0
    pts+=1.0 if _bool(p.get("hemoptysis")) else 0.0
    pts+=1.0 if _bool(p.get("cancer_active")) else 0.0
    return {"name":"WELLS_PE","score": pts, "tier2": ("likely" if pts>4 else "unlikely")}

def calc_wells_dvt(p: Dict[str,Any]):
    pts=0
    pts+=1 if _bool(p.get("cancer_active")) else 0
    pts+=1 if (_bool(p.get("paresis")) or _bool(p.get("plaster_cast"))) else 0
    pts+=1 if (_bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w"))) else 0
    pts+=1 if _bool(p.get("deep_vein_tenderness")) else 0
    pts+=1 if _bool(p.get("entire_leg_swollen")) else 0
    pts+=1 if _bool(p.get("calf_swelling_gt3cm")) else 0
    pts+=1 if _bool(p.get("pitting_edema")) else 0
    pts+=1 if _bool(p.get("collateral_nonvaricose")) else 0
    pts+=1 if _bool(p.get("prev_dvt")) else 0
    pts-=2 if _bool(p.get("alt_dx_as_likely")) else 0
    return {"name":"WELLS_DVT","score": int(pts), "tier2": ("likely" if pts>=2 else "unlikely")}

def calc_perc(p: Dict[str,Any]):
    crit = {
        "age<50": int((_pick(p,["age"],int) or 10) < 50),
        "hr<100": int((_pick(p,["hr","pulse"],float) or 0) < 100),
        "sao2>=95": int((_pick(p,["sao2","spo2"],float) or 0) >= 95),
        "no_hemoptysis": int(not _bool(p.get("hemoptysis"))),
        "no_estrogen": int(not _bool(p.get("estrogen_use"))),
        "no_surg/trauma_4w": int(not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w")))),
        "no_prior_vte": int(not _bool(p.get("prev_vte"))),
        "no_unilateral_swelling": int(not _bool(p.get("unilateral_leg_swelling"))),
    }
    return {"name":"PERC","passed": bool(all(crit.values()))}

def calc_pesi(p: Dict[str,Any]):
    age=_pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if ((_pick(p,["hr","pulse"],float) or 0) >=110) else 0
    sbp = 30 if ((_pick(p,["sbp"],float) or 200) < 100) else 0
    rr = 20 if ((_pick(p,["rr"],float) or 0) >=30) else 0
    temp = 20 if ((_pick(p,["temp"],float) or 37) < 36) else 0
    altered = 60 if ((_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if ((_pick(p,["sao2","spo2"],float) or 100) < 90) else 0
    score = age+male+cancer+hf+lung+hr+sbp+rr+temp+altered+sat
    klass = "I" if score<=65 else "II" if score<=85 else "III" if score<=105 else "IV" if score<=125 else "V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p: Dict[str,Any]):
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulm": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2"],float) or 100) < 90),
    }
    return {"name":"sPESI","score": int(sum(comps.values()))}

def calc_marburg(p: Dict[str,Any]):
    sex=(str(p.get("sex") or "")[:1]).upper(); age=_pick(p,["age"],int)
    vasc=_bool(p.get("vasc_disease")); exert=_bool(p.get("exertional")); pt_thinks=_bool(p.get("patient_assumes_cardiac"))
    palp = p.get("palpation_reproducible"); not_repro = (palp is False)
    age_sex = ((sex=="M" and age is not None and age>=55) or (sex=="F" and age is not None and age>=65))
    sc = int(bool(age_sex)) + int(vasc) + int(exert) + int(pt_thinks) + int(bool(not_repro))
    return {"name":"MARBURG","score": int(sc)}

def calc_gbs(p: Dict[str,Any]):
    score=0
    urea=_pick(p,["urea_mmol_l","urea"]); bun=_pick(p,["bun_mg_dl"])
    if urea is None and bun is not None: urea=float(bun)/2.8
    hb_gL=_pick(p,["hb_g_l"]); 
    if hb_gL is None:
        hb_gdl=_pick(p,["hb","hb_g_dl"]); 
        if hb_gdl is not None: hb_gL=hb_gdl*10.0
    sbp=_pick(p,["sbp"]); hr=_pick(p,["hr","pulse"]); male = str(p.get("sex") or "").upper().startswith("M")
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0; score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0; score += 6 if urea>25.0 else 0
    if hb_gL is not None:
        if male:   score += (1 if 120<=hb_gL<=129 else 0) + (3 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
        else:      score += (1 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
    if sbp is not None: score += (1 if 100<=sbp<=109 else 0) + (2 if 90<=sbp<=99 else 0) + (3 if sbp<90 else 0)
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p: Dict[str,Any]):
    bili=_pick(p,["bilirubin"]); alb=_pick(p,["albumin"]); inr=_pick(p,["inr"])
    asc=(p.get("ascites") or "").lower(); ence=(p.get("encephalopathy") or "").lower()
    sc=0; filled=0
    if bili is not None: sc+=(1 if bili<2 else 2 if bili<=3 else 3); filled+=1
    if alb  is not None: sc+=(1 if alb>3.5 else 2 if alb>=2.8 else 3); filled+=1
    if inr  is not None: sc+=(1 if inr<1.7 else 2 if inr<=2.3 else 3); filled+=1
    if asc:              sc+=(1 if asc.startswith("n") else 2 if asc.startswith(("mild","slight")) else 3); filled+=1
    if ence:             sc+=(1 if ence in {"none","0"} else 2 if any(x in ence for x in ["1","2","i","ii"]) else 3); filled+=1
    if filled<5: return {"name":"CHILD_PUGH","score_partial": int(sc), "class":"incomplete"}
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    if total_ca is None or albumin is None: return None
    if "mmol" in (units or "").lower():
        alb_gl = albumin if albumin>10 else albumin*10.0
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na)+(float(k) if k is not None else 0.0)) - (float(cl)+float(hco3))
    agc = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(agc,1) if agc is not None else None}

# --- Lab assessors (reuse HL7 unit converters if available; else safe fallback) ---

def assess_d_dimer(value, unit, age, pregnant=False):
    if value is None: return {"available": False}
    # Prefer canonical converter from HL7 cell if present
    conv = globals().get("_to_ddimer_mgL_feu", None)
    if conv:
        mgL = conv(value, unit); val_ug = mgL*1000.0 if mgL is not None else None
    else:
        unit_l = (unit or "").lower()
        # fallback assumes FEU; micro→mg
        if "mg/l" in unit_l: val_ug = float(value)*1000.0
        else:                val_ug = float(value)  # assume μg/L FEU
    thr = 500.0
    if age is not None and age>50 and not pregnant:
        thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr)}

# Strict ESC 0/1h for Roche hs-cTnT, backward-compatible return keys
def assess_troponin_delta(series):
    from datetime import datetime
    if not series: 
        return {"available": False}
    # normalize to ng/L and sort by time
    ser=[]
    for ts,v,u in series:
        if v is None: 
            continue
        try:
            v = float(v)
            if isinstance(u,str) and ("µg/l" in u.lower() or "ug/l" in u.lower()):
                v *= 1000.0
        except Exception:
            continue
        ser.append((ts, v))
    ser = sorted(ser, key=lambda x: (x[0] or datetime.min))
    if len(ser) < 2:
        # one sample only
        return {"available": True, "prev": ser[-1][1], "curr": ser[-1][1],
                "delta_abs": 0.0, "delta_pct": None, "flag": False,
                "dt_min": None, "rule01h": None, "reason": "needs_serial"}

    # latest sample as t1; search an exact 60-min predecessor
    t1_ts, t1 = ser[-1]
    pair = None
    for j in range(len(ser)-2, -1, -1):
        t0_ts, t0 = ser[j]
        if t0_ts is None or t1_ts is None:
            continue
        dt_min = round((t1_ts - t0_ts).total_seconds()/60)
        if dt_min == 60:
            pair = (t0_ts, t0, dt_min); break

    if pair:
        _, t0, dt = pair
        dv = t1 - t0
        dp = (abs(dv)/t0*100.0) if t0 else None
        # ESC 0/1h thresholds (Roche hs-cTnT)
        if (t0 >= 52.0) or (dv >= 5.0):
            label = "rule_in"
        elif (t0 < 12.0) and (dv < 3.0):
            label = "rule_out"
        else:
            label = "observe"
        return {"available": True, "prev": t0, "curr": t1,
                "delta_abs": dv, "delta_pct": dp, "flag": (label=="rule_in"),
                "dt_min": dt, "rule01h": label, "reason": "ok"}

    # no exact 60-min pair → compute latest delta for info, but no 0/1h label
    t0_ts, t0 = ser[-2]
    dv = t1 - t0
    dp = (abs(dv)/t0*100.0) if t0 else None
    gap = round((t1_ts - (t0_ts or t1_ts)).total_seconds()/60) if t0_ts and t1_ts else None
    return {"available": True, "prev": t0, "curr": t1,
            "delta_abs": dv, "delta_pct": dp, "flag": False,
            "dt_min": gap, "rule01h": None, "reason": "no_01h_pair"}

In [36]:
# === Phase-2 bundle helpers (lab normalization + context) ===
from typing import Dict, Any, Optional, List, Tuple

def build_labs_norm(labs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Map canonical HL7 keys to calculator-friendly names (presence-guarded).
    Does NOT mutate input; returns a new dict.
    """
    labs = dict(labs or {})
    out = dict(labs)

    # Bilirubin: µmol/L (canonical) → mg/dL (calculators)
    try:
        if "lab_bili_total_umolL" in labs and "bilirubin" not in out:
            out["bilirubin"] = float(labs["lab_bili_total_umolL"]) / 17.104
    except Exception:
        pass

    # Albumin: g/L (canonical) → g/dL (calculators)
    try:
        if "lab_albumin_gL" in labs:
            alb_gdl = float(labs["lab_albumin_gL"]) / 10.0
            out.setdefault("albumin", alb_gdl)
            out.setdefault("albumin_g_dl", alb_gdl)
    except Exception:
        pass

    # Electrolytes for anion gap; lactate for Sepsis3 (prefer explicit; fall back to VBG)
    if "na" not in out and "vbg_Na" in labs: out["na"] = labs["vbg_Na"]
    if "cl" not in out and "vbg_Cl" in labs: out["cl"] = labs["vbg_Cl"]
    if "hco3" not in out and "vbg_HCO3" in labs: out["hco3"] = labs["vbg_HCO3"]
    if "lactate" not in out and "vbg_lactate" in labs: out["lactate"] = labs["vbg_lactate"]

    return out

def _ctx_age_pregnant(vitals: Dict[str,Any], context: Optional[Dict[str,Any]]) -> tuple[Optional[int], bool]:
    context = context or {}
    age = _pick({**vitals, **context}, ["age"], int)
    pregnant = bool(context.get("pregnant", False))
    return age, pregnant


In [37]:
# === Vitals normalizer (model-free, presence-guarded) ===
import re, math
import pandas as pd
from typing import Dict, Any, Optional

_VITAL_SYNONYMS = {
    "SBP":      ["SBP","RR_sys","BP_sys","systolic","Systolic"],
    "DBP":      ["DBP","RR_dia","BP_dia","diastolic","Diastolic"],
    "HR":       ["HR","HF","heart_rate"],
    "SpO2":     ["SpO2","SpO₂","SpO2_pct","Sats"],
    "RR_rate":  ["RR_rate","AF","Resp","respiratory_rate"],  # breaths/min (not blood pressure!)
    "Temp_C":   ["Temp_C","Temp","temperature_c","T"],
    "GCS":      ["GCS"],
    "RR_str":   ["RR","RR_mmHg","RR_str"],                    # e.g. "120/80"
}

def _first(fd: Dict[str,Any], keys) -> Optional[Any]:
    for k in keys:
        if k in fd and fd[k] is not None:
            return fd[k]
    return None

def _parse_bp_pair(text: str) -> Optional[tuple]:
    m = re.search(r"(\d{2,3})\s*/\s*(\d{2,3})", str(text))
    if m: return float(m.group(1)), float(m.group(2))
    return None

def normalize_vitals(raw: Dict[str,Any], now_utc: Optional[pd.Timestamp]=None) -> Dict[str,Any]:
    """
    Returns canonical vitals independent of any model bundle:
    SBP, DBP, HR, SpO2, RR_rate, Temp_C, GCS, shock_index, hypotension_flag, hypoxia_flag, since_vitals_min.
    """
    out: Dict[str,Any] = {}
    # SBP/DBP from direct fields or "RR '120/80'"
    sbp = _first(raw, _VITAL_SYNONYMS["SBP"])
    dbp = _first(raw, _VITAL_SYNONYMS["DBP"])
    rr_pair = _first(raw, _VITAL_SYNONYMS["RR_str"])
    if (sbp is None or dbp is None) and rr_pair is not None:
        p = _parse_bp_pair(rr_pair)
        if p: sbp = sbp or p[0]; dbp = dbp or p[1]
    out["SBP"]  = float(sbp) if sbp is not None else None
    out["DBP"]  = float(dbp) if dbp is not None else None

    # Other primary vitals
    for canon, keys in [("HR","HR"),("SpO2","SpO2"),("RR_rate","RR_rate"),("Temp_C","Temp_C"),("GCS","GCS")]:
        val = _first(raw, _VITAL_SYNONYMS[canon])
        out[canon] = float(val) if val is not None else None

    # Derived, model-free
    hr  = out.get("HR"); sbp = out.get("SBP"); spo2 = out.get("SpO2")
    out["shock_index"]     = (hr/sbp) if (isinstance(hr,(int,float)) and isinstance(sbp,(int,float)) and sbp>0) else None
    out["hypotension_flag"]= (sbp is not None and sbp < 90)
    out["hypoxia_flag"]    = (spo2 is not None and spo2 < 90)

    # Age/sex NEVER used here (bias hygiene); gates don’t touch protected attrs.
    # since_vitals_min: prefer provided timestamp, else compute from 'vitals_ts' or now
    now = now_utc or pd.Timestamp.utcnow()
    ts  = raw.get("vitals_ts") or raw.get("ts")  # accept either if present
    try:
        tsv = pd.to_datetime(ts) if ts is not None else None
    except Exception:
        tsv = None
    out["since_vitals_min"] = (now - tsv).total_seconds()/60.0 if tsv is not None else None
    return out


In [38]:
# === Phase-2 bundle core scores (pure) ===
from typing import Dict, Any, Optional

def compute_phase2_scores(vitals: Dict[str,Any], labs_norm: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context = context or {}

    s_qsofa = calc_qsoFA(vitals) if 'calc_qsoFA' in globals() else calc_qsofa(vitals)  # tolerate minor naming diffs
    s_mews  = calc_mews(vitals)
    s_heart = calc_heart({**vitals, **labs_norm})
    s_grace = calc_grace_coarse({**vitals, **labs_norm})
    s_sofa  = calc_sofa_min(vitals, labs_norm)
    sirs    = calc_sirs({**vitals, **labs_norm})
    sepsis3 = sepsis3_screen(vitals, labs_norm, context, s_sofa, s_qsofa)

    wells_pe  = calc_wells_pe({**vitals, **labs_norm, **context})
    wells_dvt = calc_wells_dvt({**vitals, **labs_norm, **context})
    perc      = calc_perc({**vitals, **labs_norm, **context})
    pesi      = calc_pesi({**vitals, **labs_norm, **context})
    spesi     = calc_spesi({**vitals, **labs_norm, **context})
    marburg   = calc_marburg({**vitals, **labs_norm, **context})
    gbs       = calc_gbs({**vitals, **labs_norm, **context})
    childpugh = calc_child_pugh({**vitals, **labs_norm, **context})

    return {
        "qsofa": s_qsofa, "mews": s_mews, "heart": s_heart, "grace": s_grace, "sofa": s_sofa,
        "sirs": sirs, "sepsis3": sepsis3,
        "wells_pe": wells_pe, "wells_dvt": wells_dvt, "perc": perc,
        "pesi": pesi, "spesi": spesi,
        "marburg": marburg, "gbs": gbs, "childpugh": childpugh
    }


In [39]:
# === Phase-2 bundle (assembler) ===
from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime

def phase2_bundle(vitals: Dict[str,Any], labs: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context = context or {}
    labs_norm = build_labs_norm(labs)
    age, pregnant = _ctx_age_pregnant(vitals, context)

    # Scores
    sc = compute_phase2_scores(vitals, labs_norm, context)
    s_qsofa=sc["qsofa"]; s_mews=sc["mews"]; s_heart=sc["heart"]; s_grace=sc["grace"]; s_sofa=sc["sofa"]
    sirs=sc["sirs"]; sepsis3=sc["sepsis3"]
    wells_pe=sc["wells_pe"]; wells_dvt=sc["wells_dvt"]; perc=sc["perc"]
    pesi=sc["pesi"]; spesi=sc["spesi"]; marburg=sc["marburg"]; gbs=sc["gbs"]; childpugh=sc["childpugh"]

    # D-dimer (canonical mg/L FEU or legacy µg/L FEU)
    if "lab_ddimer_mgL_FEU" in labs_norm:
        d_val = float(labs_norm["lab_ddimer_mgL_FEU"]); d_unit = "mg/L FEU"
    else:
        d_val  = _pick(labs_norm, ["d_dimer_feu_ug_l","d_dimer","ddimer","d-dimer"])
        d_unit = labs_norm.get("d_dimer_unit") or "µg/L FEU"
    d_assess = assess_d_dimer(d_val, d_unit, age, pregnant) if d_val is not None else {"available": False}

    # Troponin Δ (legacy series only)
    series: List[Tuple[Optional[datetime], float, str]] = []
    for it in (labs_norm.get("troponin_series") or []):
        if isinstance(it, dict):
            ts=None
            if it.get("time"):
                try: ts = datetime.fromisoformat(str(it["time"]).replace("Z",""))
                except Exception: ts=None
            series.append((ts, _pick(it, ["value","val"]), it.get("unit","ng/L")))
    t_assess = assess_troponin_delta(series) if len(series) >= 2 else {"available": False}

    # Corrections
    ca_corr = corrected_calcium(
        _pick(labs_norm, ["calcium","ca","calcium_mg_dl"]),
        _pick(labs_norm, ["albumin","alb","albumin_g_dl"]),
        units="mg/dL"
    )
    ag_val  = anion_gap(
        _pick(labs_norm, ["na","sodium"]),
        _pick(labs_norm, ["cl","chloride"]),
        _pick(labs_norm, ["hco3","bicarbonate"]),
        k=_pick(labs_norm, ["k","potassium"]),
        albumin_gdl=_pick(labs_norm, ["albumin","alb","albumin_g_dl"])
    )
    # One-liners
    ones = []
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(
            f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → "
            f"{'OK' if d_assess['ok_below_thr'] else 'High'}"
        )

    # --- hs-cTnT (Roche) ESC 0/1h one-liner (strict 60 min) ---
    if t_assess.get("available"):
        r = t_assess.get("rule01h")
        if r is None:
            dt = t_assess.get("dt_min")
            ones.append(
                "hs-cTnT 0/1h: OBSERVE — " +
                ("needs serial" if dt is None else f"no 60-min pair (latest gap={int(dt)} min)")
            )
        else:
            lbl = r.replace("_","-").upper()  # RULE-IN | RULE-OUT | OBSERVE
            ones.append(
                f"hs-cTnT 0/1h: {lbl} (t0={t_assess['prev']:.0f}, t1={t_assess['curr']:.0f}, "
                f"Δ={t_assess['delta_abs']:.0f} ng/L; 60 min)"
            )

    ones.append(
        f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tier2']})  "
        f"Wells-DVT={wells_dvt['score']} ({wells_dvt['tier2']})"
    )
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(
        f"SIRS={sirs['score']}  Sepsis3: {'YES' if sepsis3['sepsis_flag'] else 'no'}  "
        f"Shock: {'YES' if sepsis3['septic_shock'] else 'no'}"
    )
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh.get("class") == "incomplete":
        ones.append("Child-Pugh incomplete (need 5/5 inputs)")
    else:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None:
        ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag_val is not None:
        ab = f", AGcorr={ag_val['ag_albumin_corrected']}" if ag_val.get("ag_albumin_corrected") is not None else ""
        ones.append(f"Anion gap={ag_val['ag']}{ab}")

    # --- build payload (add troponin timing + gate signals) ---
    out = {
        "one_liners": ones,
        "scores": {
            "qsofa": s_qsofa["score"], "mews": s_mews["score"], "heart": s_heart["score"], "grace_coarse": s_grace["score"],
            "sofa_min": s_sofa["score"], "wells_pe": wells_pe["score"], "wells_dvt": wells_dvt["score"],
            "pesi": pesi["score"], "spesi": spesi["score"], "sirs": sirs["score"],
            "sepsis3": int(sepsis3["sepsis_flag"]), "gbs": gbs["score"], "marburg": marburg["score"],
            "child_pugh": childpugh.get("score") or childpugh.get("score_partial")
        },
        "labs": {
            "d_dimer": (d_assess.get("value_ug_per_l") if d_assess.get("available") else None),
            "d_dimer_thr": d_assess.get("thr_ug_per_l"),
            "trop_prev": t_assess.get("prev"), "trop_curr": t_assess.get("curr"),
            "trop_delta": t_assess.get("delta_abs"), "trop_delta_pct": t_assess.get("delta_pct"),
            "trop_flag": t_assess.get("flag"),
            "trop_rule01h": t_assess.get("rule01h"),
            "trop_dt_min":  t_assess.get("dt_min"),
        },
    }

    # minimal advice-only signals for later gate usage
    out.setdefault("signals", {})
    out["signals"]["acs_rulein"]  = (t_assess.get("rule01h") == "rule_in")
    out["signals"]["acs_ruleout"] = (t_assess.get("rule01h") == "rule_out")

    return out


In [40]:
# ---- UI (ipywidgets) + HL7 merge ----
if getattr(CFG, "run_ui", False):
    try:
        import ipywidgets as W, pandas as pd, json
        EVENT_LOG_PATH = str(CFG.event_log_path)

        # --- Text areas ---
        vitals_in = W.Textarea(
            value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}',
            description="Vitals JSON", layout=W.Layout(width="100%", height="90px")
        )
        labs_in = W.Textarea(
            value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}',
            description="Labs JSON", layout=W.Layout(width="100%", height="130px")
        )
        ctx_in = W.Textarea(
            value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites":"mild","encephalopathy":"1-2"}',
            description="Context JSON", layout=W.Layout(width="100%", height="90px")
        )

        # --- HEART component inputs (0–2) ---
        heart_hist = W.Dropdown(options=[0,1,2], value=0, description="HEART: History")
        heart_ecg  = W.Dropdown(options=[0,1,2], value=0, description="HEART: ECG")
        heart_risk = W.Dropdown(options=[0,1,2], value=0, description="HEART: Risk")

        # --- PE / PERC flags ---
        chk_pe_most  = W.Checkbox(description="PE most likely", value=True)
        chk_dvt_signs= W.Checkbox(description="DVT signs", value=False)
        chk_immob    = W.Checkbox(description="Immobilized / recent surgery (4w)", value=False)
        chk_prev_vte = W.Checkbox(description="Prior VTE", value=False)
        chk_hemo     = W.Checkbox(description="Hemoptysis", value=False)
        chk_cancer   = W.Checkbox(description="Active malignancy", value=False)
        chk_estrogen = W.Checkbox(description="Estrogen use (PERC)", value=False)
        chk_trauma   = W.Checkbox(description="Recent trauma (4w, PERC)", value=False)
        chk_unilat   = W.Checkbox(description="Unilateral leg swelling (PERC)", value=False)

        # --- Wells-DVT flags ---
        dvt_cancer   = W.Checkbox(description="Active cancer", value=False)
        dvt_paresis  = W.Checkbox(description="Paresis / plaster cast", value=False)
        dvt_bed_surg = W.Checkbox(description="Bedridden ≥3d / surgery ≤12w", value=False)
        dvt_tender   = W.Checkbox(description="Deep vein tenderness", value=False)
        dvt_entire   = W.Checkbox(description="Entire leg swollen", value=False)
        dvt_calf3    = W.Checkbox(description="Calf swelling >3 cm", value=False)
        dvt_edema    = W.Checkbox(description="Pitting edema (symptomatic leg)", value=False)
        dvt_collat   = W.Checkbox(description="Collateral non-varicose", value=False)
        dvt_prev     = W.Checkbox(description="Previous DVT", value=False)
        dvt_alt_dx   = W.Checkbox(description="Alternative dx as likely (subtract)", value=False)

        # --- HL7 paste area ---
        hl7_in = W.Textarea(
            value='''OBX|1|NM|BILIRUBIN TOTAL||36|umol/L|||N||F|||20250820090000
OBX|2|NM|INR||1.9|||N||F|||20250820090000
OBX|3|NM|ALBUMIN||28|g/L|||N||F|||20250820090000
OBX|4|NM|D-DIMER||780|ug/L|||H||F|||20250820090000
OBX|5|NM|TROPONIN T||78|ng/L|||H||F|||20250820090000
OBX|6|NM|TROPONIN T||18|ng/L|||N||F|||20250820050000''',
            description="HL7 paste", layout=W.Layout(width="100%", height="140px")
        )

        btn_apply_hl7 = W.Button(description="Apply HL7 → Labs JSON")
        btn_compute   = W.Button(description="Compute + Log")
        out = W.Output()

        def on_apply(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_in.value or "{}")
                except Exception as e:
                    print("[labs parse error]", e)
                    return
                parsed = parse_hl7_labs(hl7_in.value or "")
                # Merge core others
                oth = parsed.get("others") or {}
                for k in ["bilirubin","inr","albumin"]:
                    if oth.get(k) is not None:
                        labs[k] = oth[k]
                # If available, set D-dimer (take newest) when missing
                dd = parsed.get("d_dimer") or []
                if dd and "d_dimer" not in labs:
                    dd_sorted = sorted(dd, key=lambda t: (t[0] or 0))
                    _, v, u = dd_sorted[-1]
                    labs["d_dimer"] = v
                    labs["d_dimer_unit"] = u
                # Always (re)write troponin_series if present
                ts = parsed.get("troponin") or []
                if ts:
                    series = []
                    for t, v, u in ts:
                        series.append({
                            "time": (t.isoformat()+"Z") if t else None,
                            "value": v, "unit": u or "ng/L"
                        })
                    labs["troponin_series"] = series
                labs_in.value = json.dumps(labs, ensure_ascii=False)
                print("HL7 merged →", {k: labs.get(k) for k in ["bilirubin","inr","albumin","d_dimer","d_dimer_unit"]})

        def on_compute(_):
            with out:
                out.clear_output()
                try:
                    vitals = json.loads(vitals_in.value or "{}")
                    labs   = json.loads(labs_in.value or "{}")
                    ctx    = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e)
                    return

                # HEART sub-scores from dropdowns
                vitals["heart_history"] = int(heart_hist.value)
                vitals["heart_ecg"]     = int(heart_ecg.value)
                vitals["heart_risk"]    = int(heart_risk.value)

                # PE / PERC + DVT flags
                ctx.update({
                    "pe_most_likely": chk_pe_most.value,
                    "dvt_signs": chk_dvt_signs.value,
                    "immobilized": chk_immob.value,
                    "recent_surgery_4w": chk_immob.value,
                    "prev_vte": chk_prev_vte.value,
                    "hemoptysis": chk_hemo.value,
                    "cancer_active": chk_cancer.value or ctx.get("cancer_active", False),
                    "estrogen_use": chk_estrogen.value,
                    "recent_trauma_4w": chk_trauma.value,
                    "unilateral_leg_swelling": chk_unilat.value,
                    # Wells-DVT details
                    "paresis": dvt_paresis.value,
                    "plaster_cast": dvt_paresis.value,
                    "bedridden_3d": dvt_bed_surg.value,
                    "surgery_12w": dvt_bed_surg.value,
                    "deep_vein_tenderness": dvt_tender.value,
                    "entire_leg_swollen": dvt_entire.value,
                    "calf_swelling_gt3cm": dvt_calf3.value,
                    "pitting_edema": dvt_edema.value,
                    "collateral_nonvaricose": dvt_collat.value,
                    "prev_dvt": dvt_prev.value,
                    "alt_dx_as_likely": dvt_alt_dx.value,
                })

                bundle = phase2_bundle(vitals, labs, ctx)

                # ML risk (only if bridge loaded)
                if "predict_one" in globals() and "FEAT" in globals():
                    feats = FEAT
                    row = {f: labs.get(f, vitals.get(f, ctx.get(f, 0.0))) for f in feats}
                    res = predict_one(row)
                    bundle["one_liners"].append(
                        f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})"
                    )
                    try:
                        _append_event({"type": "ml_score", "keys": list(row.keys()), "res": res})
                    except Exception:
                        pass
                else:
                    bundle["one_liners"].append("ML risk: (bridge not loaded)")

                _append_event({"type":"phase2_bundle","bundle": bundle})
                print("\n".join(bundle["one_liners"]))
                print("\nLogged to:", EVENT_LOG_PATH)

        btn_apply_hl7.on_click(on_apply)
        btn_compute.on_click(on_compute)

        display(W.VBox([
            W.HTML("<b>Phase-2 calculators — optimized UI (PE/DVT/PERC, PESI/sPESI, SIRS/Sepsis3, HEART, GBS, Child-Pugh)</b>"),
            W.HBox([vitals_in, labs_in]),
            W.HBox([heart_hist, heart_ecg, heart_risk]),
            W.HTML("<b>PE / PERC flags</b>"),
            W.HBox([chk_pe_most, chk_dvt_signs, chk_immob, chk_prev_vte, chk_hemo, chk_cancer]),
            W.HBox([chk_estrogen, chk_trauma, chk_unilat]),
            W.HTML("<b>Wells-DVT flags</b>"),
            W.HBox([dvt_cancer, dvt_paresis, dvt_bed_surg, dvt_tender, dvt_entire]),
            W.HBox([dvt_calf3, dvt_edema, dvt_collat, dvt_prev, dvt_alt_dx]),
            ctx_in,
            W.HBox([btn_compute, btn_apply_hl7]),
            W.HTML("<b>HL7 Labs (Trop / D-Dimer / INR / Bili / Albumin)</b>"),
            hl7_in,
            out
        ]))
    except Exception as e:
        print("Phase-2 UI unavailable:", e)

In [41]:
# --- Advice-only "Next steps" panel (ultra-minimal, no extra deps) ---
import ipywidgets as W, json, html
from pathlib import Path

def _tail_phase2_bundle(n=120):
    p = Path(str(CFG.event_log_path))
    if not p.exists():
        return None
    lines = p.read_text().splitlines()[-n:]
    for ln in reversed(lines):
        try:
            ev = json.loads(ln)
            if ev.get("type") in {"phase2_bundle", "phase2_bundle_smoke"}:
                return ev.get("bundle") or ev.get("result") or {}
        except Exception:
            pass
    return None

def _advice_from_bundle(b):
    b = b or {}
    sig  = b.get("signals") or {}
    labs = b.get("labs") or {}
    tips = []

    # ACS pathway (advice-only, requires human approval for any order/transfer)
    if sig.get("acs_rulein"):
        tips += [
            "Page cardiology (advice-only); requires human approval.",
            "Prepare cath-lab pre-checklist + contraindications form (advice-only).",
            "Confirm serial ECG monitoring is active; notify if new ischemic changes.",
        ]
    elif sig.get("acs_ruleout"):
        tips += [
            "De-prioritize ACS pathway; continue evaluation of non-ACS causes per SOP.",
            "If clinically stable and risks low, start discharge planning checklist (advice-only).",
        ]
    else:
        dt = labs.get("trop_dt_min")
        if dt is None:
            tips.append("Schedule 1h hs-cTnT exactly at 60 minutes from baseline (align phlebotomy timing).")
        else:
            tips.append(f"No exact 60-min pair (latest gap={int(dt)} min). Align labs to strict 0/1h protocol.")
        tips.append("Continue observation and supportive care per ED SOP (advice-only).")

    # Always include coordination hygiene
    tips += [
        "Broadcast a 1-line status update to ED team (advice-only).",
        "Append these suggestions to the audit log.",
    ]
    return tips

# UI
btn = W.Button(description="Next steps (advice-only)", button_style="info", icon="check")
out = W.Output()

def _on_click(_):
    with out:
        out.clear_output()
        # Prefer recomputing from current UI inputs if available
        bundle = None
        try:
            vit = json.loads(vitals_in.value)
            lab = json.loads(labs_in.value)
            ctx = json.loads(ctx_in.value)
            bundle = phase2_bundle(vit, lab, ctx)
        except Exception:
            bundle = _tail_phase2_bundle()

        if not bundle:
            print("No bundle available yet — run the calculators first.")
            return

        tips = _advice_from_bundle(bundle)
        display(W.HTML("<b>Advice-only next steps</b>"))
        items = "".join(f"<li>{html.escape(t)}</li>" for t in tips)
        display(W.HTML(f"<ul>{items}</ul>"))

btn.on_click(_on_click)
display(W.VBox([btn, out]))


In [42]:
# --- CFG-native capacity gates (no dict CONFIG) ---
from typing import Dict, Any, List

# unified getter (works for dicts or dataclasses; tolerates lower/upper keys)
def _cfg(cfg, key: str, default=None):
    for k in (key, key.lower(), key.upper()):
        try:
            return cfg.get(k, default)
        except Exception:
            if hasattr(cfg, k):
                return getattr(cfg, k)
    return default

# Optional: default ICU preference map (can override via CFG.icu_prefs later)
ICU_PREFS_DEFAULT: Dict[str, List[str]] = {
    "cardiac":     ["cap_cardio_icu","cap_cardio_surgery_icu","cap_vascular_cardiac_icu"],
    "respiratory": ["cap_internal_medicine_icu","cap_interdis_stage1","cap_interdis_stage2","cap_interdis_stage3"],
    "neuro":       ["cap_neuro_icu","cap_stroke_unit","cap_neurosurgery_icu"],
    "surgery":     ["cap_surgery_icu","cap_intermediate_care_surgery"],
}

def capacity_gatepack(fd: Dict[str, Any], cfg) -> Dict[str, Any]:
    """
    Compute capacity-related gates. Reads prefs from CFG (if present) and falls
    back to ICU_PREFS_DEFAULT. Pass CFG (dataclass), not a dict.
    """
    gates: List[str] = []
    alerts: List[str] = []

    suspected = str(fd.get("suspected_condition", "") or "").lower()
    admit     = str(fd.get("admit_decision", "") or "").upper()

    # pick the relevant capacity fields for this condition
    icu_prefs = _cfg(cfg, "icu_prefs", ICU_PREFS_DEFAULT)
    cap_fields = icu_prefs.get(suspected, ICU_PREFS_DEFAULT.get(suspected, []))

    # evaluate capacities present in the feature dict
    caps = [float(fd.get(f, 1.0) or 0.0) for f in cap_fields if f in fd]

    # rule: if any relevant capacity is 0 and the decision is ICU -> block & escalate
    if admit == "ICU" and caps and any(c <= 0.0 for c in caps):
        gates += ["ICU_BLOCKED", "ED_BOARDING_RISK", "ROUTE_PLAN_ED_TO_ICU", "TRANSFER_BLOCK_CAPACITY"]
        alerts += [f"ALERT_CAPACITY_{suspected.upper()}_ICU_0"]

    # next action policy
    next_action = "ESCALATE_CAPACITY_AND_PLAN_TRANSFER" if "ICU_BLOCKED" in gates else "PROCEED"

    return {"gates": gates, "alerts": alerts, "next_action": next_action}


In [43]:
# Force-sync all "Context JSON" textareas and the reference used by YEARS
import json, ipywidgets as W, gc

def _update_all_context_widgets(flag=True):
    new_val = None
    # If a ctx_in exists in this scope, start from it
    try:
        d = json.loads(ctx_in.value or "{}")
    except Exception:
        d = {}
    d["pregnant"] = bool(flag)
    new_val = json.dumps(d, ensure_ascii=False)

    # Update any Textarea whose description starts with "Context JSON"
    n = 0
    for obj in gc.get_objects():
        try:
            if isinstance(obj, W.Textarea) and (obj.description or "").startswith("Context JSON"):
                obj.value = new_val
                n += 1
        except Exception:
            pass
    print(f"Updated {n} Context JSON widget(s) →", new_val)

    # Make sure the YEARS cell reads the same widget reference
    global ctx_src  # used inside the YEARS cell handler
    try:
        ctx_src = ctx_in
        print("ctx_src → ctx_in (bound)")
    except NameError:
        print("ctx_in not in scope; YEARS will still read its own ctx_src if present.")

_update_all_context_widgets(flag=True)


Updated 0 Context JSON widget(s) → {"pregnant": true}
ctx_in not in scope; YEARS will still read its own ctx_src if present.


In [44]:

# === Pregnancy-adapted YEARS pathway (drop-in) ===

from datetime import datetime, timezone
import json

def assess_pregnancy_years(ctx, labs):
    # Preg-adapted YEARS: three items — DVT signs, hemoptysis, 'PE most likely'.
    # If none present → D-dimer threshold 1000 μg/L FEU; else threshold 500 μg/L FEU.
    preg = bool(ctx.get("pregnant", False))
    items = {
        "dvt_signs": bool(ctx.get("dvt_signs", False)),
        "hemoptysis": bool(ctx.get("hemoptysis", False)),
        "pe_most_likely": bool(ctx.get("pe_most_likely", False)),
    }
    count = sum(int(v) for v in items.values())
    d_val = None
    for k in ("d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"):
        if k in labs and labs[k] not in (None, ""):
            try:
                d_val = float(str(labs[k]).replace(",", "."))
                break
            except Exception:
                pass
    d_unit = (labs.get("d_dimer_unit") or "μg/L FEU").lower()
    if d_val is None:
        return {"applies": preg, "needs_imaging": True, "reason": "missing D-dimer", "items": items, "items_count": count}
    d_ug = float(d_val)*1000.0 if "mg/l" in d_unit else float(d_val)
    thr = 1000.0 if count==0 else 500.0
    ruleout = bool(d_ug < thr)
    return {
        "applies": preg, "items": items, "items_count": count,
        "d_dimer_ug_l": d_ug, "threshold_ug_l": thr, "rule_out": ruleout,
        "note": "If DVT signs present, compression ultrasound is recommended before applying YEARS."
    }

# UI that reuses Phase-2 widget inputs if present (vitals_in / labs_in / ctx_in).
try:
    _probe = vitals_in  # noqa: F401
    _has_phase2_ui = True
except NameError:
    _has_phase2_ui = False

if getattr(CFG, "run_ui", False):
    try:
        import ipywidgets as W
        if _has_phase2_ui:
            labs_src = labs_in
            ctx_src  = ctx_in
            src_note_ctx = W.HTML('<i>Using Context JSON from Phase-2 panel</i>')
            src_note_labs = W.HTML('<i>Using Labs JSON from Phase-2 panel</i>')
        else:
            labs_src = W.Textarea(value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU"}', description="Labs JSON", layout=W.Layout(width="100%", height="80px"))
            ctx_src  = W.Textarea(value='{"pregnant": true, "pe_most_likely": true, "dvt_signs": false, "hemoptysis": false}', description="Context JSON", layout=W.Layout(width="100%", height="80px"))
            src_note_ctx = src_note_labs = W.HTML('')
        out = W.Output()
        btn = W.Button(description="Compute Pregnancy YEARS + Log")
        warn = W.HTML("<small><b>Note:</b> PERC is not validated in pregnancy; use the YEARS pathway below.</small>")

        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_src.value if hasattr(labs_src, 'value') else labs_src)
                    ctx  = json.loads(ctx_src.value if hasattr(ctx_src, 'value') else ctx_src)
                except Exception as e:
                    print("[parse error]", e); return
                res = assess_pregnancy_years(ctx, labs)
                if not res.get("applies", False):
                    print("Pregnancy YEARS: not applicable (pregnant flag is false).")
                else:
                    msg = f"Pregnancy YEARS: items={res['items_count']} → D-dimer thr {int(res['threshold_ug_l'])} μg/L; value {int(res['d_dimer_ug_l'])} → "
                    msg += ("RULE-OUT" if res["rule_out"] else "IMAGING")
                    print(msg)
                try:
                    _append_event({"type":"pregnancy_years", "result": res})
                    print("Logged to:", str(CFG.event_log_path))
                except Exception as e:
                    print("[log error]", e)

        btn.on_click(_on_click)

        display(W.VBox([
            W.HTML("<b>Pregnancy-adapted YEARS pathway</b>"),
            warn,
            src_note_ctx, src_note_labs,
            btn, out
        ]))
    except Exception as e:
        print("YEARS UI unavailable:", e)

# Text-mode smoke if RUN_UI is off
if not getattr(CFG, "run_ui", False):
    labs = {"d_dimer": 780, "d_dimer_unit": "μg/L FEU"}
    ctx = {"pregnant": True, "pe_most_likely": True, "dvt_signs": False, "hemoptysis": False}
    res = assess_pregnancy_years(ctx, labs)
    msg = f"Pregnancy YEARS (text-mode): items={res.get('items_count',0)}; thr={int(res.get('threshold_ug_l',0))} μg/L; value={int(res.get('d_dimer_ug_l',0))} → "
    msg += ("RULE-OUT" if res.get("rule_out") else "IMAGING")
    print(msg)
    try:
        _append_event({"type":"pregnancy_years_smoke", "result": res})
        print("Logged to:", str(CFG.event_log_path))
    except Exception as e:
        print("[log error]", e)


Pregnancy YEARS (text-mode): items=1; thr=500 μg/L; value=780 → IMAGING
Logged to: /kaggle/working/event_log.jsonl


In [45]:
# === Safe tail of EVENT_LOG_PATH (optional) ===
import os, subprocess, shlex
from pathlib import Path
EVENT_LOG_PATH = str(CFG.event_log_path)
if Path(EVENT_LOG_PATH).exists():
    try:
        subprocess.run(["tail","-n","20",EVENT_LOG_PATH], check=False)
    except Exception as e:
        print("tail failed (ignored):", e)
else:
    print("No event log yet (will be created on first write).")

{"ts": "2025-08-31T17:40:16.292747+00:00", "type": "pregnancy_years_smoke", "result": {"applies": true, "items": {"dvt_signs": false, "hemoptysis": false, "pe_most_likely": true}, "items_count": 1, "d_dimer_ug_l": 780.0, "threshold_ug_l": 500.0, "rule_out": false, "note": "If DVT signs present, compression ultrasound is recommended before applying YEARS."}}


In [46]:
# === Med Plan Import + Checks (terse) ===
from __future__ import annotations
from typing import Dict, Any, List, Optional
from pathlib import Path
from datetime import datetime, timezone
import json, re

CFG = globals().get("CFG")
CONFIG = globals().get("CONFIG", {})
RUN_MEDS = getattr(CFG, "run_meds", CONFIG.get("RUN_MEDS", False))
RUN_UI   = getattr(CFG, "run_ui",   CONFIG.get("RUN_UI",   False))
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/med_rules.json")
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/allergies.json")
EVENT_LOG = Path(str(getattr(CFG, "event_log_path", "/kaggle/working/event_log.jsonl")))

if "_append_event" not in globals():
    def _append_event(ev: Dict[str, Any]):
        EVENT_LOG.parent.mkdir(parents=True, exist_ok=True); EVENT_LOG.touch(exist_ok=True)
        with EVENT_LOG.open("a", encoding="utf-8") as fp:
            fp.write(json.dumps({"ts": datetime.now(timezone.utc).isoformat(), **(ev or {})}, ensure_ascii=False) + "\n")

# --- map (brand/generic → atc,class) ---
_ATC_MAP: Dict[str, Dict[str, str]] = {
    # coumarins
    "marcumar":{"atc":"B01AA04","class":"coumarin"},
    "phenprocoumon":{"atc":"B01AA04","class":"coumarin"},
    "warfarin":{"atc":"B01AA03","class":"coumarin"},
    # antibiotics / inhibitors
    "amoxicillin":{"atc":"J01CA04","class":"penicillin"},
    "erythromycin":{"atc":"J01FA01","class":"macrolide"},
    "clarithromycin":{"atc":"J01FA09","class":"macrolide"},
    "azithromycin":{"atc":"J01FA10","class":"macrolide"},
    "ketoconazole":{"atc":"J02AB02","class":"azole_strong"},
    "itraconazole":{"atc":"J02AC02","class":"azole_strong"},
    "voriconazole":{"atc":"J02AC03","class":"azole_strong"},
    "posaconazole":{"atc":"J02AC04","class":"azole_strong"},
    "fluconazole":{"atc":"J02AC01","class":"azole_moderate"},
    "ciprofloxacin":{"atc":"J01MA02","class":"fqn_qt"},
    "levofloxacin":{"atc":"J01MA12","class":"fqn_qt"},
    "moxifloxacin":{"atc":"J01MA14","class":"fqn_qt"},
    # DOACs
    "apixaban":{"atc":"B01AF02","class":"doac_xa"},
    "rivaroxaban":{"atc":"B01AF01","class":"doac_xa"},
    "edoxaban":{"atc":"B01AF03","class":"doac_xa"},
    "dabigatran":{"atc":"B01AE07","class":"doac_ii"},
    # statins
    "simvastatin":{"atc":"C10AA01","class":"statin_cyp3a"},
    "atorvastatin":{"atc":"C10AA05","class":"statin_cyp3a"},
    "rosuvastatin":{"atc":"C10AA07","class":"statin_non3a"},
    "pravastatin":{"atc":"C10AA03","class":"statin_non3a"},
    # QT agents
    "haloperidol":{"atc":"N05AD01","class":"qt_drug"},
    "citalopram":{"atc":"N06AB04","class":"qt_drug"},
    "escitalopram":{"atc":"N06AB10","class":"qt_drug"},
    "amiodarone":{"atc":"C01BD01","class":"qt_drug"},
    "sotalol":{"atc":"C07AA07","class":"qt_drug"},
    "ziprasidone":{"atc":"N05AE04","class":"qt_drug"},
    "quetiapine":{"atc":"N05AH04","class":"qt_drug"},
    # serotonin cluster
    "sertraline":{"atc":"N06AB06","class":"ssri"},
    "fluoxetine":{"atc":"N06AB03","class":"ssri"},
    "linezolid":{"atc":"J01XX08","class":"mao_i"},
    "tramadol":{"atc":"N02AX02","class":"serotonergic"},
    # nitrates / PDE-5
    "nitroglycerin":{"atc":"C01DA02","class":"nitrate"},
    "isosorbide dinitrate":{"atc":"C01DA08","class":"nitrate"},
    "isosorbide mononitrate":{"atc":"C01DA14","class":"nitrate"},
    "sildenafil":{"atc":"G04BE03","class":"pde5i"},
    "tadalafil":{"atc":"G04BE08","class":"pde5i"},
    "vardenafil":{"atc":"G04BE09","class":"pde5i"},
    # metformin / contrast
    "metformin":{"atc":"A10BA02","class":"metformin"},
    "iodinated contrast":{"atc":"—","class":"contrast"},
    "kontrastmittel":{"atc":"—","class":"contrast"},
    "jodhaltig":{"atc":"—","class":"contrast"},
    # kidney/AKI cluster
    "ramipril":{"atc":"C09AA05","class":"ace"},
    "lisinopril":{"atc":"C09AA03","class":"ace"},
    "valsartan":{"atc":"C09CA03","class":"arb"},
    "hydrochlorothiazid":{"atc":"C03AA03","class":"diuretic"},
    "hct":{"atc":"C03AA03","class":"diuretic"},
    "furosemid":{"atc":"C03CA01","class":"diuretic"},
    # keep ibuprofen for completeness
    "ibuprofen":{"atc":"M01AE01","class":"nsaid"},
    # others used earlier
    "spironolacton":{"atc":"C03DA01","class":"aldosterone_antagonist"},
    "metoprolol":{"atc":"C07AB02","class":"beta_blocker"},
}

# --- ED rules (class/name aware) ---
RULES: List[Dict[str, Any]] = [
    {"a_class":"coumarin","b":"amoxicillin","severity":"moderate",
     "action":"Continue if indicated; increase monitoring.",
     "monitor":["INR","Quick","PTT","Fibrinogen","anti-Xa (if applicable)","ROTEM (if available)"],
     "message":"Coumarin + amoxicillin → ↑ anticoagulant effect."},
    {"a_class":"coumarin","b_class":"macrolide","severity":"major",
     "action":"Prefer alt ABx; if used, monitor closely.","monitor":["INR","Quick","PTT","Fibrinogen"],
     "message":"Coumarin + macrolide → bleeding risk."},

    {"a_class":"doac_xa","b_class":"azole_strong","severity":"major",
     "action":"Avoid/consult; consider dose adjust/alt.","monitor":["anti-Xa (calibrated)","renal","bleeding"],
     "message":"Xa-DOAC + strong azole inhibitor."},
    {"a_class":"doac_xa","b_class":"macrolide","severity":"moderate",
     "action":"Caution; review dose/indication.","monitor":["anti-Xa (calibrated)","renal"],
     "message":"Xa-DOAC + macrolide (P-gp/CYP3A)."},
    {"a_class":"doac_ii","b_class":"macrolide","severity":"moderate",
     "action":"Caution; consider dTT/ECT.","monitor":["dTT/ECT","renal"],
     "message":"Dabigatran + macrolide (P-gp)."},
    {"a_class":"doac_xa","b":"amiodarone","severity":"moderate",
     "action":"Caution; consider dose adjust.","monitor":["anti-Xa (calibrated)"],
     "message":"Xa-DOAC + amiodarone."},
    {"a_class":"doac_ii","b":"amiodarone","severity":"moderate",
     "action":"Caution; consider dTT/ECT.","monitor":["dTT/ECT"],
     "message":"Dabigatran + amiodarone."},

    {"a_class":"statin_cyp3a","b_class":"macrolide","severity":"major",
     "action":"Hold statin now and 3–5d after.","monitor":["CK if myalgia","LFTs if symptomatic"],
     "message":"CYP3A statin + macrolide → rhabdo risk."},
    {"a_class":"statin_cyp3a","b_class":"azole_strong","severity":"major",
     "action":"Avoid; switch to non-3A statin.","monitor":["CK","LFTs if symptomatic"],
     "message":"CYP3A statin + strong azole."},

    {"a_class":"macrolide","b_class":"qt_drug","severity":"moderate",
     "action":"ECG baseline/24–48h; correct K+/Mg2+.","monitor":["ECG","K+","Mg2+"],
     "message":"Additive QT prolongation."},
    {"a_class":"fqn_qt","b_class":"qt_drug","severity":"moderate",
     "action":"ECG baseline/24–48h; correct K+/Mg2+.","monitor":["ECG","K+","Mg2+"],
     "message":"Additive QT prolongation."},

    {"a_class":"metformin","b_class":"contrast","severity":"moderate",
     "action":"Hold around contrast; re-check eGFR ~48h.","monitor":["eGFR 48h"],
     "message":"Metformin + contrast → lactic acidosis risk in AKI."},

    {"a_class":"nitrate","b_class":"pde5i","severity":"major",
     "action":"Contraindicated: avoid (24h sildenafil/vardenafil, 48h tadalafil).","monitor":[],
     "message":"Severe hypotension risk."},

    {"a_class":"ssri","b":"tramadol","severity":"moderate",
     "action":"Avoid if possible; monitor for toxicity.","monitor":["vitals","neuro (clonus, agitation)"],
     "message":"SSRI + tramadol → serotonin syndrome risk."},

    {"a_class":"ace","b_class":"diuretic","severity":"info",
     "action":"If NSAID added, prefer alternative; monitor renal.","monitor":["Creatinine","K+"],
     "message":"ACE/ARB + diuretic + NSAID → AKI risk."},
]

_ROUTES = ["p.o.","po","i.v.","iv","i.m.","im","s.c.","sc","inhalativ","topisch","nasal","otic","ophthalmic"]
_FREQ_WORDS = ["morgens","mittags","abends","nachts"]
_FREQ_PAT = re.compile(r"\b(\d+)[-/.](\d+)[-/.](\d+)\b")
_DOSE_PAT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(mg|g|mcg|µg|ml|IE|Einheiten)\b", re.I)

def _normalize_name(s:str)->str:
    s=s.strip().lower().replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss")
    return re.sub(r"[^a-z0-9]+"," ",s).strip()

def _map_to_atc(name_norm:str)->Dict[str,Any]:
    for k,meta in _ATC_MAP.items():
        if k in name_norm: return {"name_norm":k,**meta}
    return {"name_norm":name_norm,"atc":None,"class":None}

def parse_med_line(line:str)->Optional[Dict[str,Any]]:
    raw=line.strip()
    if not raw or raw.startswith("#"): return None
    name=raw.split(",")[0].split("  ")[0]; nn=_normalize_name(name)
    dose_val=dose_unit=freq=route=None; m=_DOSE_PAT.search(raw)
    if m: dose_val=float(m.group(1).replace(",", ".")); dose_unit=m.group(2).lower()
    m=_FREQ_PAT.search(raw)
    if m: freq=f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    else:
        w=[w for w in _FREQ_WORDS if w in raw.lower()]
        if w: freq=",".join(w)
    low=raw.lower()
    for r in _ROUTES:
        if r in low: route=r; break
    prn=bool(re.search(r"\b(prn|bei bedarf)\b",raw,re.I))
    meta=_map_to_atc(nn)
    return {"raw":raw,"name":name.strip(),"name_norm":meta["name_norm"],"atc":meta["atc"],"class":meta["class"],
            "dose_value":dose_val,"dose_unit":dose_unit,"frequency":freq,"route":route,"prn":prn}

def parse_med_text(text:str)->List[Dict[str,Any]]:
    return [rec for ln in text.splitlines() if (rec:=parse_med_line(ln))]

def load_rules(path:str)->List[Dict[str,Any]]:
    p=Path(path)
    if p.exists():
        try:
            js=json.loads(p.read_text(encoding="utf-8"))
            if isinstance(js,list): return js
        except Exception: pass
    return list(RULES)

def load_allergies(path:str)->List[str]:
    p=Path(path)
    if p.exists():
        try:
            js=json.loads(p.read_text(encoding="utf-8"))
            if isinstance(js,dict) and isinstance(js.get("allergies"),list):
                return [str(x) for x in js["allergies"]]
        except Exception: pass
    return []

def check_interactions(meds:List[Dict[str,Any]], rules:List[Dict[str,Any]])->List[Dict[str,Any]]:
    out=[]; names={m["name_norm"] for m in meds}; classes={m["class"] for m in meds if m.get("class")}
    for r in rules:
        a_ok=(("a" in r and r["a"] in names) or ("a_class" in r and r["a_class"] in classes))
        b_ok=(("b" in r and r["b"] in names) or ("b_class" in r and r["b_class"] in classes))
        if a_ok and b_ok: out.append({"type":"med_interaction_warning",**r})
    return out

def check_allergies(meds:List[Dict[str,Any]], terms:List[str])->List[Dict[str,Any]]:
    out=[]; t=[t.lower() for t in terms]
    for m in meds:
        if any(x in m["name"].lower() for x in t): out.append({"type":"med_allergy_warning","med":m["name"],"match":"name"}); continue
        if any("penicillin" in x for x in t):
            if m.get("class")=="penicillin" or m["name"].lower().endswith("cillin"):
                out.append({"type":"med_allergy_warning","med":m["name"],"match":"class_penicillin"})
    return out

def ocr_file(path:str)->Optional[str]:
    p=Path(path)
    if not p.exists(): return None
    try:
        from PIL import Image; import pytesseract
        if p.suffix.lower() in (".png",".jpg",".jpeg",".tif",".tiff"): return pytesseract.image_to_string(Image.open(p))
        if p.suffix.lower()==".pdf":
            try:
                from pdf2image import convert_from_path
                return "\n".join(pytesseract.image_to_string(img) for img in convert_from_path(str(p)))
            except Exception: return None
    except Exception: return None
    return None

def emit_meds_events(patient_id:str, meds:List[Dict[str,Any]], warnings:List[Dict[str,Any]]):
    _append_event({"type":"med_plan_import","patient_id":patient_id,"n_meds":len(meds)})
    for m in meds: _append_event({"type":"med_entry","patient_id":patient_id,**m})
    for w in warnings: _append_event({**w,"patient_id":patient_id})

print("Medication import/checks enabled." if RUN_MEDS else
      "Deferred… set CONFIG['RUN_MEDS']=True or CFG.run_meds=True to enable medication import & checks.")

if RUN_UI and RUN_MEDS:
    try:
        import ipywidgets as W, pandas as pd
        from IPython.display import display
        pid=W.Text(description="Patient ID",placeholder="e.g., UKE-12345")
        pth=W.Text(description="Scan path",placeholder="/mnt/data/scan.pdf (optional)")
        ocr_btn=W.Button(description="Run OCR"); go=W.Button(description="Parse & Check",button_style="primary")
        ta=W.Textarea(description="Plan text",layout=W.Layout(width="100%",height="180px")); out=W.Output()
        def on_ocr(_):
            with out: out.clear_output(); txt=ocr_file(pth.value.strip()); print("OCR complete." if txt else "OCR unavailable or failed."); 
            ta.value = txt or ta.value
        def on_go(_):
            with out:
                out.clear_output(); text=ta.value.strip()
                if not text: print("No text provided."); return
                meds=parse_med_text(text)
                rules=load_rules(CONFIG.get("MED_RULES_PATH")); alg=load_allergies(CONFIG.get("ALLERGIES_PATH"))
                warns=check_interactions(meds,rules)+check_allergies(meds,alg); emit_meds_events(pid.value or "DEMO",meds,warns)
                display(pd.DataFrame(meds) if meds else pd.DataFrame()); 
                print("\nWarnings:"); display(pd.DataFrame(warns) if warns else pd.DataFrame({"info":["none"]}))
        ocr_btn.on_click(on_ocr); go.on_click(on_go)
        display(W.VBox([pid,pth,W.HBox([ocr_btn,go]),ta,out]))
    except Exception as e:
        print("Meds UI unavailable:", e)


Medication import/checks enabled.


In [47]:
# === Gate-pack merger (pure; TTL=max; order-preserving dedup) ===
from typing import Dict, Any

def _merge_ttl_max(a: Dict[str,int]|None, b: Dict[str,int]|None) -> Dict[str,int]:
    out = dict(a or {})
    for k, v in (b or {}).items():
        try:
            out[k] = max(out.get(k, 0), int(v))
        except Exception:
            out[k] = out.get(k, 0)
    return out

def merge_gatepacks(*packs: Dict[str, Any]) -> Dict[str, Any]:
    out: Dict[str, Any] = {"gates": [], "priority": 0, "next_action": "", "explain": [], "ttl": {}}
    for p in packs:
        if not p: 
            continue
        out["gates"].extend(p.get("gates", []))
        out["priority"] = max(out["priority"], int(p.get("priority", 0)))
        if p.get("next_action"):
            out["next_action"] = p["next_action"]
        if p.get("explain"):
            out["explain"].extend(p["explain"])
        out["ttl"] = _merge_ttl_max(out["ttl"], p.get("ttl"))
    # order-preserving dedup
    seen, dedup = set(), []
    for g in out["gates"]:
        if g not in seen:
            seen.add(g); dedup.append(g)
    out["gates"] = dedup
    return out

def spc_gatepack(metric_value: float, limits: dict):
    if spc_flag(metric_value, limits):
        return {"gates": ["SPC_OUT_OF_CONTROL"],
                "alerts": ["ALERT_SPC_SHIFT"],
                "next_action": "ESCALATE_INVESTIGATE_ROOT_CAUSE"}
    return {"gates": [], "alerts": [], "next_action": "PROCEED"}


In [48]:
# === Gate-engine (composes packs; ICU & BGA fallback if packs absent) ===
# CFG-native fallback ICU gate pack
from typing import Dict, Any

def _fallback_pack_icu(fd: Dict[str, Any], cfg) -> Dict[str, Any]:
    """
    Simple capacity rule for ICU:
      - If admit_decision == 'ICU' and capacity is low, emit gates/alerts.
      - Thresholds come from cfg (if present), else sensible defaults.
    cfg may be a dict or a dataclass; values looked up via _cfg().
    """
    # thresholds (do not mutate cfg)
    th_ok    = float(_cfg(cfg, "TH_ICU_CAP_OK",    0.15))  # ≥15% = OK
    th_warn  = float(_cfg(cfg, "TH_ICU_CAP_WARN",  0.05))  # 5–15% = warning
    th_block = float(_cfg(cfg, "TH_ICU_CAP_BLOCK", 0.00))  # ≤0% = blocked

    cap_raw  = fd.get("cap_internal_medicine_icu", None)
    cap      = None if cap_raw is None else float(cap_raw)
    need_icu = str(fd.get("admit_decision", "")).upper() == "ICU"

    gates, alerts = [], []

    if need_icu and cap is not None:
        if cap <= th_block:
            gates += ["ICU_BLOCKED", "ED_BOARDING_RISK", "ROUTE_PLAN_ED_TO_ICU", "TRANSFER_BLOCK_CAPACITY"]
            alerts += ["ALERT_CAPACITY_ICU_0"]
            next_action = "ESCALATE_CAPACITY_AND_PLAN_TRANSFER"
        elif cap < th_warn:
            gates += ["ICU_SEVERELY_TIGHT"]
            alerts += ["ALERT_CAPACITY_ICU_SEVERE"]
            next_action = "ESCALATE_CAPACITY_COORDINATION"
        elif cap < th_ok:
            gates += ["ICU_TIGHT"]
            alerts += ["ALERT_CAPACITY_ICU_TIGHT"]
            next_action = "COORDINATE_ADMISSION"
        else:
            next_action = "PROCEED"
    else:
        # not an ICU admit or no capacity reading → neutral
        next_action = "PROCEED"

    return {"gates": gates, "alerts": alerts, "next_action": next_action}


def _fallback_pack_bga(fd: Dict[str,Any], CONFIG: Dict[str,Any]) -> Dict[str,Any]:
    # thresholds from your ops policy
    pH  = fd.get("vbg_pH")
    K   = fd.get("vbg_K")
    Na  = fd.get("vbg_Na")
    Lac = fd.get("vbg_lactate")
    gates, explain, ttl = [], [], {}
    # pH
    if pH is not None:
        if float(pH) > 7.55:
            gates.append("BGA_ALKALOSIS_CRITICAL"); explain.append(f"pH {pH} > 7.55")
        elif float(pH) >= 7.5:
            gates.append("BGA_ALKALOSIS_HIGH"); explain.append(f"pH {pH} ≥ 7.5")
    # Potassium
    if K is not None:
        kf = float(K)
        if kf >= 6.5: gates.append("BGA_HYPERKALAEMIA_CRITICAL"); explain.append(f"K⁺ {kf} ≥ 6.5 mmol/L")
        elif kf >= 6.0: gates.append("BGA_HYPERKALAEMIA_HIGH"); explain.append(f"K⁺ {kf} ≥ 6.0 mmol/L")
        elif kf < 2.5: gates.append("BGA_HYPOKALAEMIA_CRITICAL"); explain.append(f"K⁺ {kf} < 2.5 mmol/L")
    # Sodium
    if Na is not None:
        naf = float(Na)
        if naf <= 125: gates.append("BGA_DYSNATRAEMIA_CRITICAL"); explain.append(f"Na⁺ {naf} ≤ 125 mmol/L")
        elif (naf <= 129) or (naf >= 145): gates.append("BGA_DYSNATRAEMIA"); explain.append(f"Na⁺ {naf} outside 130–144 mmol/L")
    # Lactate (typical critical)
    if Lac is not None and float(Lac) >= 4.0:
        gates.append("BGA_LACTATE_CRITICAL"); explain.append(f"Lactate {Lac} ≥ 4.0 mmol/L")
    # escalate alerts for any critical
    if any(g.endswith("CRITICAL") for g in gates):
        gates += ["ALERT_NURSE","ALERT_DOC","ALERT_ATTENDING"]
        ttl.update({"ALERT_NURSE":300,"ALERT_DOC":300,"ALERT_ATTENDING":300})
    if not gates:
        return {"gates": [], "priority": 0, "next_action": "", "explain": [], "ttl": {}}
    return {
        "gates": gates,
        "priority": 2 if any(g.endswith("CRITICAL") for g in gates) else 1,
        "next_action": "Address critical VBG derangements now." if any(g.endswith("CRITICAL") for g in gates) else "",
        "explain": explain,
        "ttl": ttl
    }

def gate_engine(fd: Dict[str,Any], CONFIG: Dict[str,Any]) -> Dict[str,Any]:
    """
    Compose gate-packs if present; otherwise use ICU+BGA fallbacks to guarantee safety signals.
    No I/O. Pure function.
    """
    packs = []
    # Prefer your real pack producers if they’re defined in the notebook/module
    for name in [
        # add your real producers here if present, e.g.:
        "ops_capacity_gatepack",           # ICU/boarding pack
        "bga_gatepack",                    # VBG/electrolytes pack
        "abdomen_guardrails_gatepack",     # imaging + labs policy
        "overdue_gatepack",                # last_seen & vitals overdues
        "troponin_policy_gatepack",        # troponin timing/delta policy
    ]:
        fn = globals().get(name)
        if callable(fn):
            try:
                packs.append(fn(fd, CONFIG))
            except Exception as e:
                # non-fatal: continue composing with other packs
                packs.append({"gates":[], "priority":0, "next_action":"", "explain":[f"{name} failed: {e}"], "ttl":{}})

    # If key packs are missing, add safe fallbacks
    if not any("ICU_" in g for p in packs for g in p.get("gates", [])):
        packs.append(_fallback_pack_icu(fd, CONFIG))
    if not any(g.startswith("BGA_") for p in packs for g in p.get("gates", [])):
        packs.append(_fallback_pack_bga(fd, CONFIG))

    # Merge all packs
    return merge_gatepacks(*packs)


In [49]:
# Gate dedup + TTL max policy
p1 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":120}}
p2 = {"gates":["ALERT_NURSE"], "priority":0, "next_action":"", "explain":[], "ttl":{"ALERT_NURSE":300}}
m  = merge_gatepacks(p1, p2)
assert m["ttl"].get("ALERT_NURSE") == 300
print("TTL_MAX_OK")

TTL_MAX_OK


In [50]:
# == Tiny patch: canonical refresh_sop_registry + parse_hl7_labs ==
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional
import re, json

def refresh_sop_registry(CONFIG: dict, base_url: Optional[str]="https://sop-notaufnahme.de/sop/")->Dict[str,Any]:
    # Fetch product pages, grab first PDF, write canonical CSV
    out_csv = Path(str(CFG.sop_registry_path)); pdf_dir = Path(CONFIG["DATA_ROOT"])/"sop_pdfs"; pdf_dir.mkdir(parents=True, exist_ok=True)
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":f"missing libs: {e}", "csv": str(out_csv)}
    found=saved=errors=0; recs=[]
    try:
        soup = BeautifulSoup(requests.get(base_url, timeout=15).text, "html.parser")
        links = sorted({a["href"] for a in soup.select('a[href*="/product/"]') if a["href"].startswith("http")})
        for u in links:
            try:
                ps = BeautifulSoup(requests.get(u, timeout=15).text, "html.parser")
                title = (ps.find(["h1","h2"]) or ps.find("title") or "").get_text(strip=True)
                sid = re.sub(r"[^a-z0-9]+","-", (title or u).lower()).strip("-")
                a = ps.select_one('a[href$=".pdf"]'); pdf_path=""
                if a:
                    try:
                        fp = pdf_dir/f"{sid}.pdf"
                        with requests.get(a["href"], stream=True, timeout=30) as r:
                            r.raise_for_status()
                            with open(fp,"wb") as f:
                                for c in r.iter_content(8192): 
                                    if c: f.write(c)
                        pdf_path=str(fp); saved+=1
                    except Exception: errors+=1; pdf_path=a["href"]
                recs.append({"sop_id":sid,"title":title or sid,"pdf_path":pdf_path,"version":"","status":"fetched" if pdf_path else "linked","keywords":"","checklist":"","source_url":u})
                found+=1
            except Exception: errors+=1
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":str(e), "csv": str(out_csv)}
    import pandas as pd
    cols=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"]
    df = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(columns=cols)
    for c in cols:
        if c not in df.columns: df[c]=""
    for r in recs:
        m = (df["sop_id"].astype(str)==r["sop_id"])
        if m.any():
            for k,v in r.items():
                if k in df.columns and (pd.isna(df.loc[m,k]).all() or str(df.loc[m,k].iloc[0]).strip()=="" or k in ["pdf_path","status","source_url"]):
                    df.loc[m,k]=v
        else:
            df = pd.concat([df, pd.DataFrame([r])], ignore_index=True)
    df[cols].to_csv(out_csv, index=False)
    return {"found":found,"saved":saved,"errors":errors,"csv":str(out_csv),"dir":str(pdf_dir)}

def parse_hl7_labs(txt: str)->Dict[str,Any]:
    trops, dd, oth = [], [], {"bilirubin":None,"inr":None,"albumin":None}
    for ln in re.split(r'[\r\n]+', (txt or "").strip()):
        if not ln.startswith("OBX|"): continue
        p = ln.split("|"); obx3, obx5, obx6, obx14 = (p+[None]*15)[3], (p+[None]*15)[5], (p+[None]*15)[6], (p+[None]*15)[14]
        ts=None; m=re.search(r'(\d{8})(\d{6})?', obx14 or "")
        if m:
            try: ts=datetime.strptime(m.group(1)+(m.group(2) or "000000"), "%Y%m%d%H%M%S")
            except: pass
        try: v=float(str(obx5).replace(",", "."))
        except:
            mm=re.search(r'[-+]?\d*\.?\d+', str(obx5)); v=float(mm.group(0)) if mm else None
        if v is None: continue
        u=(obx6 or ""); name=(obx3 or ln).upper()
        if "TROP" in name: trops.append((ts,v,u or "ng/L"))
        if any(k in name for k in ["D-DIMER","DDIMER","D DIMER"]): dd.append((ts,v,u or "μg/L FEU"))
        if "BILIRUBIN" in name or "BILI" in name:
            if oth["bilirubin"] is None: oth["bilirubin"] = v/17.1 if u.lower() in {"umol/l","µmol/l"} else v
        if "INR" in name and oth["inr"] is None: oth["inr"]=v
        if "ALBUMIN" in name and oth["albumin"] is None: oth["albumin"]= v/10.0 if u.lower() in {"g/l","g l","gl"} else v
    return {"troponin":trops,"d_dimer":dd,"others":oth}


In [51]:
# === Preflight Check (simplified) ===

def preflight(verbose=True):
    """ED Pipeline system readiness check"""
    
    failures = []
    log = print if verbose else lambda *args: None
    
    def check(name, condition, error_msg=""):
        """Simple pass/fail checker"""
        if condition:
            log(f"[✓] {name}")
            return True
        else:
            failures.append(f"[x] {name}" + (f" — {error_msg}" if error_msg else ""))
            return False
    
    # Core system components
    check("CONFIG present", "CONFIG" in globals())
    check("CONFIG frozen", "CONFIG" in globals() and (hasattr(CONFIG, "_FrozenConfig") or isinstance(CONFIG, dict)))
    check("_append_event present", "_append_event" in globals())
    
    # Essential functions
    required_funcs = ["run_ui", "refresh_sop_registry", "parse_hl7_labs", "phase2_bundle",
                     "_load_icu_status", "_compute_next_bed_eta"]
    for func in required_funcs:
        check(f"{func} present", func in globals(), f"{func} missing")
    
    # Phase2 bundle smoke test
    try:
        # Try UI inputs first, fallback to test data
        import json
        if all(x in globals() for x in ("vitals_in", "labs_in", "ctx_in")):
            vitals = json.loads(vitals_in.value or "{}")
            labs = json.loads(labs_in.value or "{}")
            ctx = json.loads(ctx_in.value or "{}")
        else:
            # Fallback test data
            vitals = {"rr": 24, "sbp": 95, "gcs": 14, "hr": 120, "temp": 38.6, "sao2": 97}
            labs = {"d_dimer": 780, "creatinine": 1.8, "platelets": 95, "albumin": 2.8}
            ctx = {"suspected_infection": True, "pe_most_likely": True}
        
        bundle = phase2_bundle(vitals, labs, ctx)
        check("bundle one-liners non-empty", 
              isinstance(bundle.get("one_liners"), list) and len(bundle["one_liners"]) > 3)
              
    except Exception as e:
        failures.append(f"[x] bundle smoke test — {e}")
    
    # Event logging
    try:
        import pathlib
        log_path = pathlib.Path(str(CFG.event_log_path))
        check("Event log path exists or creatable", log_path.parent.exists())
        _append_event({"type": "preflight_ping", "ok": True})
        check("Event append works", True)
    except Exception as e:
        failures.append(f"[x] Event logging — {e}")
    
    # Optional ML components
    if "predict_one" in globals() and "FEAT" in globals():
        check("predict_one wired", True)
    else:
        log("[-] ML wrapper not loaded (ok for now)")
    
    # Results
    if failures:
        log("\nRESULT: FAIL")
        for failure in failures:
            log(failure)
        return False
    else:
        log("\nRESULT: PASS")
        return True

# Auto-run preflight check
preflight()

[✓] CONFIG present
[✓] CONFIG frozen
[✓] _append_event present
[✓] run_ui present
[✓] refresh_sop_registry present
[✓] parse_hl7_labs present
[✓] phase2_bundle present
[✓] _load_icu_status present
[✓] _compute_next_bed_eta present
[✓] bundle one-liners non-empty
[✓] Event log path exists or creatable
[✓] Event append works
[✓] predict_one wired

RESULT: PASS


True

In [52]:
# --- Phase-1 quickcheck (overlay; runs only if you call it) ---
from pathlib import Path
import inspect, pandas as pd

def phase1_quickcheck(config) -> dict:
    CFG = dict(config)           # do NOT mutate CONFIG
    CFG["RUN_UI"] = False
    CFG["RUN_PIPELINE"] = False

    # Ensure CSV/dir semantics without side effects
    for k in ("QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH",
              "EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"):
        p = Path(CFG[k]); (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)

   
    # 1) Moves log writes (no None-indexing thanks to read() patch)
    t = TrackerService.from_config(CFG)
    t.log_move("__smoke__", "", "ZZ")
    moves_ok = Path(CFG["EQUIPMENT_MOVES_LOG_PATH"]).exists()

    # 2) SOP auto-pull surface (offline-safe summary)
    try:
        sop_ok = isinstance(refresh_sop_registry(CFG, base_url="http://127.0.0.1:9/offline"), dict)
    except Exception:
        sop_ok = False

    # 3) Adjustable vitals threshold present (split-aware; no regex)
    try:
        ui_src = ""
        if "run_ui_header_controls" in globals():
            ui_src = inspect.getsource(run_ui_header_controls)
        elif "run_ui" in globals():
            ui_src = inspect.getsource(run_ui)
        ui_ok = "Vitals overdue (min)" in ui_src
    except Exception:
        ui_ok = False

    return {"moves": moves_ok, "sop": sop_ok, "ui_vitals_threshold": ui_ok}

# Example opt-in usage:
# res = phase1_quickcheck(CONFIG); print(res); assert all(res.values()), res


In [53]:
# one-ping-per-episode (arrival-only)
from datetime import datetime, timezone

FLOW = dict(monitor_paused=False, episode=0, notified_ep=None, current_service="ED")

def orderly(event, dest=None, now=None):
    t = now or datetime.now(timezone.utc)
    if event == "PICKUP":
        FLOW["episode"] += 1
        FLOW["monitor_paused"] = True
        if dest: FLOW["current_service"] = dest
        FLOW["notified_ep"] = None        # reset notifier for this episode
        # no nurse ping here
    elif event == "DROPOFF_BACK_ED":
        FLOW["current_service"] = "ED"
        # send exactly one arrival ping per episode
        if FLOW.get("notified_ep") != FLOW["episode"]:
            print(f"[NURSE] {t:%H:%M}Z Patient returned to ED — please resume monitoring.")
            FLOW["notified_ep"] = FLOW["episode"]
    elif event == "RESUME":
        FLOW["monitor_paused"] = False    # optional: no ping
    else:
        pass

# demo:
# orderly("PICKUP", dest="MRI")
# orderly("DROPOFF_BACK_ED")
# orderly("DROPOFF_BACK_ED")  # (no second ping)
# orderly("RESUME")


In [54]:
# --- Fixed verifier for adjustable vitals threshold (no CONFIG mutation) ---
import inspect, re

src_ui  = ""
src_hdr = ""
try:
    src_ui  = inspect.getsource(run_ui)
except Exception:
    pass
try:
    src_hdr = inspect.getsource(run_ui_header_controls)
except Exception:
    pass

combined = (src_ui or "") + "\n" + (src_hdr or "")

# Simple substring checks
has_label  = "Vitals overdue (min)" in combined
has_number = "number_input" in combined and ("vitals" in combined.lower())

# Corrected regex (match a number_input call whose argument list mentions 'vitals')
regex_ok = bool(re.search(r'number_input\([^)]*vitals', combined, flags=re.I))

ui_vitals_adjustable = has_label or (has_number and regex_ok)

print("UI_VITALS_INPUT_OK:", ui_vitals_adjustable)
print("Found in:", ("run_ui_header_controls" if "Vitals overdue (min)" in (src_hdr or "") else "run_ui") if ui_vitals_adjustable else "NOT FOUND")


UI_VITALS_INPUT_OK: True
Found in: run_ui_header_controls


In [55]:
# --- Shadowing diff for tracker_core & edtracker package ---
import sys, hashlib
from pathlib import Path

def sha16(p: Path):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

candidates = []

# Where Python *would* import "tracker_core" from if those dirs were on sys.path
for base in map(Path, sys.path):
    mod = base / "tracker_core.py"
    if mod.is_file():
        candidates.append(mod)

# Known dataset mounts you pasted (may or may not be on sys.path)
extra = [
    Path("/kaggle/input/tracker-core/tracker_core.py"),
    Path("/kaggle/working/tracker_core.py"),  # in case a shim file exists
]
for p in extra:
    if p.is_file() and p not in candidates:
        candidates.append(p)

print("# tracker_core candidates on disk")
for p in candidates:
    try:
        print("·", p, "sha=", sha16(p))
    except Exception:
        print("·", p, "(unreadable)")

# Also show the package file actually used by edtracker
import importlib, inspect
pkg = importlib.import_module("edtracker.core.tracker_core")
print("\n# edtracker.core.tracker_core actual file")
print(inspect.getfile(pkg))


# tracker_core candidates on disk
· /kaggle/input/tracker-core/tracker_core.py sha= afa6c40d0ca3117e

# edtracker.core.tracker_core actual file
/kaggle/input/tracker-core/tracker_core.py


In [56]:
def calculate_operational_stress(spc_alerts, current_occupancy, baseline_occupancy):
    stress_score = 0.0
    
    # Base stress from occupancy
    if baseline_occupancy > 0:
        occupancy_ratio = current_occupancy / baseline_occupancy
        stress_score = max(0, min(1, (occupancy_ratio - 0.8) / 0.4))
    
    # Additional stress from SPC alerts
    for alert in spc_alerts:
        if alert['severity'] == 'high':
            stress_score = min(1.0, stress_score + 0.3)
        elif alert['severity'] == 'moderate':
            stress_score = min(1.0, stress_score + 0.1)
            
    return stress_score

def adjust_mlp_threshold(base_threshold, operational_stress):
    # Lower thresholds during high stress = more sensitive alerts
    if operational_stress > 0.7:
        return base_threshold * 0.6
    elif operational_stress > 0.4:
        return base_threshold * 0.8
    else:
        return base_threshold

In [57]:
# === NO-SURPRISES GUARDS ===

# Use config guards everywhere (prevents NameError & unintended runs)
def _on(flag): return bool(CONFIG.get(flag, False))

def run_icu_constraints(state):
    if not _on("RUN_PIPELINE"): return {}
    mod = _try_import("icu_constraints") or _try_import("modeling_icu_constraints")
    if mod and hasattr(mod,"compute_icu_flags"):
        try: return dict(mod.compute_icu_flags(state))
        except Exception: return {}
    return {}

def mesh_route_actions(state, actions):
    if not _on("RUN_PIPELINE"): return actions
    mod = _try_import("agent_mesh") or _try_import("ed_agent_mesh")
    if mod and hasattr(mod,"route"):
        try: return list(mod.route(state, actions))
        except Exception: return actions
    return actions

def trainer_fit_critic(critic, samples, y):
    if not _on("RUN_PIPELINE"): return critic
    mod = _try_import("trainer") or _try_import("ed_trainer")
    if mod and hasattr(mod,"fit_critic"):
        try: return mod.fit_critic(critic, samples, y)
        except Exception: return critic
    return critic

# Single event appender (choose one timestamp style; this is UTC with 'Z')
def _append_event(ev: dict):
    import json, datetime as _dt, os
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(str(CFG.event_log_path), "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")


In [58]:
# Validation Framework: simple utilities for cross-validation, FPR monitoring, and sensitivity analysis

import pandas as pd
import numpy as np
from typing import Dict, Tuple, List

def evaluate_detections(detections: pd.Series, outbreaks: pd.Series) -> Dict[str, float]:
    """
    detections: boolean Series indexed by time
    outbreaks: boolean Series indexed by time (ground truth)
    Returns precision/recall/FPR/TNR/TP/FP/FN/TN
    """
    d = detections.reindex(outbreaks.index).fillna(False).astype(bool)
    y = outbreaks.astype(bool)
    tp = int(((d==True) & (y==True)).sum())
    fp = int(((d==True) & (y==False)).sum())
    fn = int(((d==False) & (y==True)).sum())
    tn = int(((d==False) & (y==False)).sum())
    precision = tp / (tp+fp) if (tp+fp)>0 else 0.0
    recall = tp / (tp+fn) if (tp+fn)>0 else 0.0
    fpr = fp / (fp+tn) if (fp+tn)>0 else 0.0
    tnr = tn / (fp+tn) if (fp+tn)>0 else 0.0
    return {"TP":tp,"FP":fp,"FN":fn,"TN":tn,"precision":precision,"recall":recall,"fpr":fpr,"tnr":tnr}

def time_kfold_indices(index: pd.DatetimeIndex, k: int = 5) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    """Split a time index into k contiguous folds."""
    if not isinstance(index, pd.DatetimeIndex):
        index = pd.to_datetime(index)
    sorted_idx = index.sort_values()
    n = len(sorted_idx)
    folds = []
    sizes = [n//k + (1 if i < n%k else 0) for i in range(k)]
    start = 0
    for size in sizes:
        test_idx = sorted_idx[start:start+size]
        train_idx = sorted_idx.difference(test_idx)
        folds.append((train_idx, test_idx))
        start += size
    return folds

def cross_validate_surges(series: pd.Series, outbreaks: pd.Series, build_detector, k: int = 5):
    """
    build_detector: function (train_series) -> detector with .fit(train_series) and .predict(test_series)->bool Series
    Returns list of fold metrics and their average.
    """
    folds = time_kfold_indices(series.index, k=k)
    metrics = []
    for train_idx, test_idx in folds:
        train_s = series.reindex(train_idx).dropna()
        test_s = series.reindex(test_idx).dropna()
        det = build_detector(train_s)
        # Fit using class defined earlier
        det.fit(train_s)
        # Simple rule: signal when value outside UCL/LCL
        bl = det._baseline
        test_detect = (test_s > bl.ucl) | (test_s < bl.lcl)
        m = evaluate_detections(test_detect, outbreaks.reindex(test_idx).fillna(False))
        metrics.append(m)
    # Average
    avg = {k: float(np.mean([m[k] for m in metrics])) for k in metrics[0]}
    return metrics, avg

def sensitivity_analysis(series: pd.Series, outbreaks: pd.Series, sigma_grid=(2.0, 2.5, 3.0, 3.5), min_samples_grid=(30, 50, 80)):
    results = []
    for sg in sigma_grid:
        for ms in min_samples_grid:
            det = SPCProcessControl(sigma=sg, min_samples=ms, lookback_weeks=12)
            try:
                det.fit(series)
                bl = det._baseline
                detect = (series > bl.ucl) | (series < bl.lcl)
                m = evaluate_detections(detect, outbreaks.reindex(series.index).fillna(False))
                m.update({"sigma": sg, "min_samples": ms})
                results.append(m)
            except Exception as e:
                results.append({"sigma": sg, "min_samples": ms, "error": repr(e)})
    return pd.DataFrame(results)

In [59]:
# Demo arrivals generator (for PoC): default 300 patients per day
import pandas as pd
import numpy as np

def demo_arrivals(start="2025-01-01", periods=60, freq="D", mean_daily=300, noise_sd=15, seed=0):
    """
    Generate a synthetic arrivals time series.
    freq="D" (daily) or "H" (hourly). If hourly, daily mean is spread evenly.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start=start, periods=periods, freq=freq)
    if freq.upper().startswith("H"):
        mean = mean_daily / 24.0
    else:
        mean = mean_daily
    vals = rng.normal(loc=mean, scale=noise_sd, size=len(idx))
    vals = np.maximum(0, vals).astype(float)
    return pd.Series(vals, index=idx, name="arrivals")

In [60]:
# --- Utilities: event logging (required only if any cell calls _append_event) ---
from pathlib import Path
from datetime import datetime, timezone
import json
def _append_event(ev: dict):
    p = Path(str(CFG.event_log_path))
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **(ev or {})}
    with p.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")


In [61]:
# --- Enable flags on CFG (post-definitions) ---
from dataclasses import replace as _replace

try:
    # Build kwargs ONLY for fields that actually exist on the Settings dataclass
    fields = getattr(CFG, "__dataclass_fields__", {}) or {}
    desired = {
        "run_ui": True,
        "run_spc": True,
        "run_gate": True,
        "run_meds": True,
        "skip_model_discovery": True,
    }
    kwargs = {k: v for k, v in desired.items() if k in fields}
    if kwargs:
        CFG = _replace(CFG, **kwargs)
        print("CFG feature flags enabled →", {k: getattr(CFG, k, None) for k in kwargs})
    else:
        print("No matching CFG fields to toggle; relying on CONFIG only.")
except NameError:
    print("CFG not defined yet; feature flags will rely on CONFIG only.")


CFG feature flags enabled → {'skip_model_discovery': True}


In [62]:
# --- Feature wiring checks (gate, SPC, UI) ---
_ok = True

# gate-engine
if "gate_engine" in globals():
    try:
        getattr(gate_engine, "evaluate", None)
        print("[✓] gate_engine present")
    except Exception as e:
        _ok = False
        print("[x] gate_engine problem:", e)
else:
    _ok = False
    print("[x] gate_engine missing")

# SPC
if ("run_spc" in globals()) or ("spc_dashboard" in globals()):
    print("[✓] SPC hooks present")
else:
    _ok = False
    print("[x] SPC entrypoints missing")

# UI
if "run_ui" in globals():
    print("[✓] run_ui present")
else:
    _ok = False
    print("[x] run_ui missing")

print("WIRING:", "OK" if _ok else "ISSUES")


[✓] gate_engine present
[✓] SPC hooks present
[✓] run_ui present
WIRING: OK


In [63]:
# --- Force notebook-safe SPC if Streamlit isn't installed ---
import importlib.util as _ilu
_HAS_STREAMLIT = _ilu.find_spec("streamlit") is not None

def _run_spc_notebook():
    import pandas as pd
    from IPython.display import display
    print("SPC — notebook fallback (no Streamlit)")
    for name in ("ed","dx","meds"):
        df = globals().get(name)
        if isinstance(df, pd.DataFrame) and not df.empty:
            print(f"\n{name.upper()} sample:"); display(df.head(50))

if not _HAS_STREAMLIT:
    # override any existing run_spc that imports streamlit
    run_spc = _run_spc_notebook


In [64]:
# --- Streamlit-aware SPC/UI launcher ---
import importlib.util as _ilu
_HAS_STREAMLIT = _ilu.find_spec("streamlit") is not None

def _run_spc_notebook():
    import pandas as pd
    from IPython.display import display
    print("SPC — notebook fallback (no Streamlit)")
    for name in ("ed","dx","meds"):
        df = globals().get(name)
        if isinstance(df, pd.DataFrame) and not df.empty:
            print(f"\n{name.upper()} sample:"); display(df.head(50))

# Prefer real SPC if Streamlit exists; else notebook fallback
if "run_spc" not in globals():
    if "spc_dashboard" in globals() and _HAS_STREAMLIT:
        run_spc = spc_dashboard
    else:
        run_spc = _run_spc_notebook

try:
    if CONFIG.get("RUN_SPC", False):
        run_spc()  # safe: uses notebook fallback if Streamlit missing
        print("SPC started ✓")

    if CONFIG.get("RUN_UI", False):
        if _HAS_STREAMLIT and "run_ui" in globals():
            _append_event({"type":"ui_start"})
            _ = run_ui(tracker=tracker, get_state=get_state, get_actions=get_actions, critic=critic)
            print("UI started (Streamlit) ✓")
        else:
            print("UI skipped (Streamlit not available in this job).")
except Exception as e:
    print("UI/SPC start failed:", repr(e))
    raise


SPC — notebook fallback (no Streamlit)
SPC started ✓
UI skipped (Streamlit not available in this job).


In [65]:
# --- Health check aligned to 'everything on' ---
must_haves = {
    "predict_one": 'predict_one' in globals(),
    "gate_engine": 'gate_engine' in globals(),
    "run_spc_or_dashboard": ('run_spc' in globals()) or ('spc_dashboard' in globals()),
    "run_ui": 'run_ui' in globals(),
    "_append_event": '_append_event' in globals(),
}

for k, v in must_haves.items():
    print(f"[{'✓' if v else 'x'}] {k}")

print("\nRESULT:", "PASS" if all(must_haves.values()) else "FAIL")


[✓] predict_one
[✓] gate_engine
[✓] run_spc_or_dashboard
[✓] run_ui
[✓] _append_event

RESULT: PASS
